In [1]:
# Commented out the below line for installation of pandas if it is already installed
!python3 -m pip install pandas
import numpy as np
import pandas as pd

# Define file path
# file_path = "C:\\Users\\Zixuan\\Downloads\\MinFin V1.0.0kc.xlsm"
file_path = "data/MINFin Energy 251203 test v5.xlsm"
# Load the Excel file and check available sheet names
xls = pd.ExcelFile(file_path)
sheet_names = xls.sheet_names
# Display sheet names
sheet_names

You should consider upgrading via the 'C:\Users\Zixuan\.pyenv\pyenv-win\versions\3.11.0b4\python3.exe -m pip install --upgrade pip' command.


C:\Users\Zixuan\AppData\Local\Temp\ipykernel_24944\3160594014.py:4: DeprecationWarning: 
Pyarrow will become a required dependency of pandas in the next major release of pandas (pandas 3.0),
(to allow more performant data types, such as the Arrow string type, and better interoperability with other libraries)
but was not found to be installed on your system.
If this would cause problems for you,
please provide us feedback at https://github.com/pandas-dev/pandas/issues/54466
        
  import pandas as pd


['Definitions',
 'Visualisation Dashboard',
 'High Level Dashboard',
 'Technology Disag (S1)',
 'Technology Disag (S2)',
 'Financing Baseline',
 'Funding Baseline',
 'Investment Needs',
 'New Infrastructure (Input)']

In [2]:
import sys
sys.modules.pop("MinFin", None)  # clean the cache of MinFin module, in case it is changed between runs.
from MinFin import load_excel_data
# MinFin is a Python module designed specifically for this MINFin model
# We import functions from it   
# df_param_constraints, df_financing_baseline, df_funding_baseline, df_scenarios, df_currencies, df_technologies, df_technologies_classification =  load_excel_data(file_path)


# df_technologies.head(5) # show the first 5 rows of the technologies sheet
result = load_excel_data(file_path)
locals().update(result)
# result

In [3]:
df_technologies.head(20)

,Name,Description,Technology,Classification,Sector
0,PWRBIO001,Biomass power plant,Biomass,Generation Renewable,Power
1,PWRCOA001,Coal power plant,Coal,Generation Fossil-Fuel,Power
2,PWRCSP001,CSP - Without storage,CSP,Generation Renewable,Power
3,PWRCSP002,CSP - With storage,CSP,Generation Renewable,Power
4,PWRDIST,Electricity Distribution,Distribution,Distribution Infrastructure,Power
5,PWRGEO,Geothermal power plant,Geothermal,Generation Renewable,Power
6,PWRHYD001,Hydropower plant - Large dam (>100MW),Hydropower,Generation Renewable,Power
7,PWRHYD002,Hydropower plant - Medium (10-100MW),Hydropower,Generation Renewable,Power
8,PWRHYD003,Hydropower plant - Small (<10MW),Hydropower,Generation Renewable,Power
9,PWRHYD004,Hydropower plant - Off-grid,Direct Hydro,Generation Renewable,Power


In [4]:
class BaseDefinition:
    def __init__(self, name: str, description: str):
        self.name = name
        self.description = description
    
    def __repr__(self):
        return f"{self.__class__.__name__}(name='{self.name}', description='{self.description}')"

class ParameterConstraint(BaseDefinition):
    pass

class FinancingBaseline(BaseDefinition):
    pass

class FundingBaseline(BaseDefinition):
    pass

class Scenario(BaseDefinition):
    pass

class Currency:
    def __init__(self, code: str, currency: str):
        self.code = code
        self.currency = currency
    
    def __repr__(self):
        return f"Currency(code='{self.code}', currency='{self.currency}')"

class Technology:
    def __init__(self, name: str, technology: str, description: str, classification: str):
        self.name = name
        self.description = description
        self.classification = classification
        self.technology = technology
    
    def __repr__(self):
        return f"Technology(name='{self.name}', tech='{self.technology}', description='{self.description}', classification='{self.classification}')"

# Creating a list of objects from extracted data
def load_definitions_from_dataframe(df, cls):
    return [cls(row["Name"], row["Description"]) for _, row in df.iterrows()]

def load_currencies_from_dataframe(df):
    return [Currency(row["Code"], row["Currency"]) for _, row in df.iterrows()]

def load_technologies_from_dataframe(df):
    return [Technology(row["Name"], row["Technology"], row["Description"], row["Classification"]) for _, row in df.iterrows()]

# Example usage:
constraints = load_definitions_from_dataframe(df_param_constraints, ParameterConstraint)
baselines = load_definitions_from_dataframe(df_financing_baseline, FinancingBaseline)
fundings = load_definitions_from_dataframe(df_funding_baseline, FundingBaseline)
scenarios = load_definitions_from_dataframe(df_scenarios, Scenario)
currencies = load_currencies_from_dataframe(df_currencies)
technologies = load_technologies_from_dataframe(df_technologies.dropna())

In [5]:
df_technologies_classification.head(5)

,Technology,Classification
0,Biomass,Generation Renewable
1,Geothermal,Generation Renewable
2,Solar PV,Generation Renewable
3,CSP,Generation Renewable
4,Hydropower,Generation Renewable


In [6]:
df_param_constraints.head(5)
df_investment_needs.head(5)
# df_financing_baseline.head(5)
# df_funding_baseline.head(7)
# df_scenarios.head(5)
# df_currencies.head(5)

,Name,Description
0,Carbon Credit Price,The parameter carbon credit price refers to t...
1,Carbon Price,The parameter carbon price refers to the cost ...
2,CO2 Emissions,The parameter CO2 emissions refers to the tota...
3,Electricity Production,The parameter electricity production refers to...
4,Fixed Cost,The parameter fixed cost refers to the ongoing...


In [7]:
# dropna 并不会改变原 DataFrame，除非指定 inplace=True 或赋值回去
consumer_segments.replace(['', ' ', None], np.nan, inplace=True)
# consumer_segments.dropna(subset=["Type"],how='all', inplace=True)
consumer_segments


,Name,Currency,Type,Offtaker
0,Generation,NaN,NaN,NaN
1,National Generation FiT,USD,Generation,Export
2,National Generation PPA,GHS,Generation,Transmission
3,MiniGrid Generation,GHS,Generation,Transmission
4,Mining Generation,GHS,Generation,Distribution
5,Industry Generation,GHS,Generation,Transmission
6,Transmission,NaN,NaN,NaN
7,Export Sale Price,USD,Transmission,Distribution
8,NaN,NaN,NaN,NaN
9,NaN,NaN,NaN,NaN


In [8]:

target_categories = ["Generation", "Transmission", "Distribution", "Exports"]

# 1. 识别标题行
is_header = consumer_segments['Name'].isin(target_categories) & consumer_segments['Type'].isna()

# 2. 填充分类标签
consumer_segments['Category'] = consumer_segments['Name'].where(is_header).ffill()

# 3. 提取非标题且不为空的行，只取 Category 和 Name 列
# 这样每个 segment 都会有自己独立的一行

organized_offtaker = consumer_segments[~is_header & consumer_segments['Name'].notna()][['Category', 'Name',"Currency"]].reset_index(drop=True)

organized_offtaker

,Category,Name,Currency
0,Generation,National Generation FiT,USD
1,Generation,National Generation PPA,GHS
2,Generation,MiniGrid Generation,GHS
3,Generation,Mining Generation,GHS
4,Generation,Industry Generation,GHS
5,Transmission,Export Sale Price,USD
6,Distribution,Commercial,USD
7,Distribution,Residential,USD
8,Distribution,Industrial,USD
9,Distribution,Wholesale,USD


In [9]:
df_technologies_classification.head(5)

,Technology,Classification
0,Biomass,Generation Renewable
1,Geothermal,Generation Renewable
2,Solar PV,Generation Renewable
3,CSP,Generation Renewable
4,Hydropower,Generation Renewable


In [10]:
df_param_constraints.head(5)
df_investment_needs.head(5)
# df_financing_baseline.head(5)
# df_funding_baseline.head(7)
# df_scenarios.head(5)
# df_currencies.head(5)

,Name,Description
0,Carbon Credit Price,The parameter carbon credit price refers to t...
1,Carbon Price,The parameter carbon price refers to the cost ...
2,CO2 Emissions,The parameter CO2 emissions refers to the tota...
3,Electricity Production,The parameter electricity production refers to...
4,Fixed Cost,The parameter fixed cost refers to the ongoing...


# Read and process the Tech Disag sheet

In [11]:
tech_to_class_map = df_technologies_classification.set_index('Technology')['Classification'].to_dict()

# 定义每个技术的起始行
tech_start_rows = {
    'Biomass': 83,
    'CSP': 154,
    'Geothermal': 225,
    'Hydropower': 296,
    'Direct Hydro': 367,
    'Nuclear': 438,
    'Solar PV': 509,
    'Direct Solar': 580,
    'Imports': 651,
    'Wind': 722,
    'Coal': 793,
    'Gas': 864,
    'Direct Oil': 935,
    'Oil': 1006,
    'Transmission': 1077,
    'Distribution': 1148,
    'Energy Exports': 1219,
    
}

sheet_name = "Technology Disag (S1)"
df_full = pd.read_excel(file_path, sheet_name=sheet_name, header=None)

# 定义字段的相对位置（相对于每个技术的起始行）
# 例如：Total Grant Amount 在 Biomass 的第 105 行，相对位置 = 105 - 83 = 22
field_relative_positions = {
    # Cashflows 部分
    'total_grant_amount': {'offset': 22, 'unit': 'Million USD'},  # 105 - 83 = 22
    'ppa_currency': {'offset': 25, 'unit': 'Currency'},           # 108 - 83 = 25
    'ppa_contracted_generation': {'offset': 26, 'unit': 'GWh/Year'},  # 109 - 83 = 26
    'ppa_standard_offtaker_share': {'offset': 27, 'unit': '%'},  # 110 - 83 = 27
    'ppa_direct_offtaker_tariff': {'offset': 29, 'unit': 'USD/kWh'},  # 112 - 83 = 29
    'ppa_standard_tariff': {'offset': 30, 'unit': 'USD/kWh'},    # 113 - 83 = 30
    'ppa_contracted_capacity': {'offset': 31, 'unit': 'MW'},      # 114 - 83 = 31
    'ppa_capacity_fee': {'offset': 32, 'unit': 'Million USD/MW'}, # 115 - 83 = 32
    'ppa_penalty_tariff': {'offset': 33, 'unit': 'USD/kWh'},     # 116 - 83 = 33
    
    # Redispatch Compensation
    'redispatch_compensation_price': {'offset': 35, 'unit': 'USD/kWh'}, # 118 - 83 = 35
        
    # Share of Offtake
    # 'national_generation_fit_share': {'offset': 38, 'unit': '%'},  # 121 - 83 = 38
    # 'national_generation_ppa_share': {'offset': 39, 'unit': '%'},  # 122 - 83 = 39
    # 'minigrid_generation_share': {'offset': 40, 'unit': '%'},     # 123 - 83 = 40
    # 'mining_generation_share': {'offset': 41, 'unit': '%'},        # 124 - 83 = 41
    # 'industry_generation_share': {'offset': 42, 'unit': '%'},      # 125 - 83 = 42
    
    # Sale Prices
    'corporate tax': {'offset': 55, 'unit': ''},  # 138 - 83 = 44
    'receivables': {'offset': 57, 'unit': 'Million USD'},  # 140 - 83 = 45
    'liabilities': {'offset': 58, 'unit': 'Million USD'},      # 141 - 83 = 48
}

def generate_share_configs(tech_name,organized_offtaker, share_start_offset=38, price_start_offset=44):
    """
    动态生成 share 和 sale_price 的配置
    :param share_start_offset: share 数据相对于锚点的偏移量
    :param price_start_offset: sale_price 数据相对于锚点的偏移量 (根据您的 Excel 实际情况调整)
    """
    configs = {}
    filtered_df = organized_offtaker[organized_offtaker['Category'].str.lower().apply(lambda x: x in tech_to_class_map[tech_name].lower())]
    # print(tech_to_class_map[tech_name])
    for i, row in filtered_df.iterrows():
        if str(row.get('Category', '')).strip().lower() != 'exports':
            # process non-exports categories
            name = row['Name']
            # 生成基础 slug，例如 "national_generation_fit"
            base_slug = name.lower().replace(" ", "_").replace(":", "").replace("-", "_")
            
            # 1. 生成 Share 配置
            share_key = f"{base_slug}_share_{row['Currency']}"
            configs[share_key] = {
                'offset': share_start_offset + i,
                'unit': '%',
                'type': 'share'
            }
            
            # 2. 生成 Sale Price 配置
            price_key = f"{base_slug}_sale_price_{row['Currency']}"
            configs[price_key] = {
                'offset': price_start_offset + i,
                'unit': 'GHS/kWh', # 或者从 row['Currency'] 获取
                'type': 'price'
            }
        
    return configs

# 使用示例
# 假设 organized_segments 是您之前得到的包含所有 segment 的 DataFrame


def extract_tech_data_relative(df_full, sheet_name, tech_name, start_row, 
                                field_positions):
    """
    使用相对位置提取技术数据，只提取输入字段（黄色背景）
    """    
    tech_data = {}
    
    # 只提取输入字段（黄色背景）
    for field_name, field_info in field_positions.items():
        absolute_row = start_row + field_info['offset'] - 1
        if absolute_row < len(df_full):
            values = df_full.fillna(0).iloc[absolute_row, 2:].values
            tech_data[field_name] = {
                'values': values,
                'unit': field_info['unit']
            }
    
    return tech_data

all_tech_data = {}
for tech_name, start_row in tech_start_rows.items():
    offtake_share_configs = generate_share_configs(tech_name,organized_offtaker)
    tech_fields = field_relative_positions | offtake_share_configs
    tech_data = extract_tech_data_relative(
        df_full, 
        sheet_name, 
        tech_name, 
        start_row,
        tech_fields,
    )
    all_tech_data[tech_name] = tech_data

field_relative_positions
all_tech_data["Energy Exports"].keys()
organized_offtaker

c:\Users\Zixuan\AppData\Local\Programs\Python\Python312\Lib\site-packages\openpyxl\worksheet\_reader.py:329: UserWarning: Conditional Formatting extension is not supported and will be removed
  warn(msg)
c:\Users\Zixuan\AppData\Local\Programs\Python\Python312\Lib\site-packages\openpyxl\worksheet\_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)


,Category,Name,Currency
0,Generation,National Generation FiT,USD
1,Generation,National Generation PPA,GHS
2,Generation,MiniGrid Generation,GHS
3,Generation,Mining Generation,GHS
4,Generation,Industry Generation,GHS
5,Transmission,Export Sale Price,USD
6,Distribution,Commercial,USD
7,Distribution,Residential,USD
8,Distribution,Industrial,USD
9,Distribution,Wholesale,USD


In [12]:
def convert_tech_data_to_dataframes(all_tech_data, column_names=None):
    """
    Convert the extracted technology data to a DataFrame, one DataFrame per technology
    
    Parameters:
    -----------
    all_tech_data : dict：{'Biomass': {'field1': {'values': [...], 'unit': '...'}, ...}, ...}
    column_names : list, optional    
    """
    tech_dataframes = {}
    
    for tech_name, tech_data in all_tech_data.items():
        # 提取所有字段的值
        data_dict = {field_name: field_info['values'] 
                    for field_name, field_info in tech_data.items()}
        
        # 确定列数和列名
        max_cols = max(len(v) for v in data_dict.values()) if data_dict else 0
        cols = column_names[:max_cols] if column_names else range(max_cols)
        
        # 截取数据并创建 DataFrame
        df = pd.DataFrame({k: v[:len(cols)] for k, v in data_dict.items()}, index=cols)
        tech_dataframes[tech_name] = df
    
    return tech_dataframes
# 使用示例
years = list(range(2025, 2071))
tech_dataframes = convert_tech_data_to_dataframes(all_tech_data,years)


 
# 如果知道列名（比如年份）
# years = [2025, 2026, 2027, ...]  # 你的年份列表
# tech_dataframes = convert_tech_data_to_dataframes(all_tech_data, column_names=years)

# 访问
biomass_df = tech_dataframes['Biomass']
csp_df = tech_dataframes['CSP']

## Read the Input Sheet

In [13]:
pd.set_option('future.no_silent_downcasting', True)# This is to avoid the warning of downcasting the data t

df_input_full = pd.read_excel(file_path, sheet_name="New Infrastructure (Input)")#, engine="openpyxl")

df_input_full.head(5)


,MINFin,Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,...,Unnamed: 216,Unnamed: 217,Unnamed: 218,Unnamed: 219,Unnamed: 220,Unnamed: 221,Unnamed: 222,Unnamed: 223,Unnamed: 224,Unnamed: 225
0,New Infrastructure - Inputs,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Scenario Name 1 (More ambitious):,NaN,Net Zero,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Scenario Name 2 (Less ambitious):,NaN,Least Cost,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,Incremental,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [14]:
from MinFin import input_extractor
# df_input_full
net_zero_extrator = input_extractor('net_zero')
net_zero_summary = net_zero_extrator.get_totals(df_input_full)
full_cap_cost_net_zero, df_ffe_net_zero, df_elec_production_nz= input_extractor.load_osemosys_input('net_zero',df_input_full)

#Display sample results
net_zero_summary.head(3)
df_elec_production_nz


,PWRBIO001,PWRCOA001,PWRCSP001,PWRCSP002,PWRDIST,PWRGEO,PWRHYD001,PWRHYD002,PWRHYD003,PWRHYD004,...,PWROHC004,PWRSOL001,PWRSOL001S,PWRSOL002,PWRSOL002S,PWRTRN,PWRTRNEXP,PWRTRNIMP,PWRWND001,PWRWND001S
Year,,,,,,,,,,,,,,,,,,,,,
2015,0.001,0,0,0,35.962576,0,21.2155,0,0,0,...,0,0.0152,0,0,0,41.66396,0,1.245,0,0
2016,23,0,0,0,39.386048,0,20.512,0,0,0,...,0,0.0968,0,0,0,45.519155,0,3.215,0,0
2017,0.0004,0,0,0,44.673572,0,20.652,0,0,0,...,0,0.152,0,0,0,51.504725,0,1.5642,0,0
2018,0.0004,0,0,0,49.122633,0,22.1,0,0,0,...,0,0.155,0,0,0,56.496975,0,0.7542,0,0
2019,0.0004,0,0,0,53.062384,0,26.950305,0,0,0,...,0,0.2531,0,0,0,60.880755,0,0.654,0,0
2020,0.001,0,0,0,59.306654,0,27.564,0,0,0,...,0,0.285,0,0,0,67.88111,0,0.352,0,0
2021,0.001,0,0,0,64.387126,0,27.5012,0,0,0,...,0,0.542,0,0,0,73.607425,0,0.2012,0,0
2022,0.001568,0,0,0,84.244899,0,30.025,0,0,0,...,0,22.64752,0,0,0,96.193093,0,0.1752,0,0
2023,0.001568,0,0,0,89.188341,0,30.955775,0,0,0,...,0,26.589369,0,0,0,101.7154,0,0.325,0,0


In [15]:
df_captial_cost = input_extractor.load_block_for('net_zero',df_input_full,'capital_cost')
df_ffe = net_zero_extrator.load(df_input_full,'ffe')
df_captial_cost

,Year,PWRBIO001,PWRCOA001,PWRCSP001,PWRCSP002,PWRDIST,PWRGEO,PWRHYD001,PWRHYD002,PWRHYD003,...,PWROHC004,PWRSOL001,PWRSOL001S,PWRSOL002,PWRSOL002S,PWRTRN,PWRTRNEXP,PWRTRNIMP,PWRWND001,PWRWND001S
0,2015,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0.000004,0,0
1,2016,0,0,0,0,0,0,0,0,0,...,0,20.622568,0,0,0,0,0.000009,0.00001,0,0
2,2017,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0.000004,0.000005,0,0
3,2018,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0.000008,0.000002,0,0
4,2019,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0.000016,0.000002,0,0
5,2020,0,0,0,0,0,0,108.20941,0,0,...,0,0,0,0,0,0,0.000021,0.000001,0,0
6,2021,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0.00002,0.000001,0,0
7,2022,0,0,0,0,0,0,433.93467,0,0,...,0,6210.1089,0,0,6210.1089,0,0.000025,0.000001,0,0
8,2023,0,0,0,0,0,0,164.11847,0,0,...,460.31225,1067.6386,6210.1089,6210.1089,1067.6386,6210.1089,6210.1089,0.000002,0,6210.1089
9,2024,0,0,0,0,203.22078,203.22078,169.20614,203.22078,203.22078,...,467.21633,0,1067.6386,1067.6386,0,1067.6386,1067.6386,0.000001,0,1067.6386


In [16]:
df_ffe.head(5)

,IMPBIO Power,IMPCOA Power,IMPHFO Power,IMPLPG Power,IMPNGS Power,IMPOIL Power,IMPURN Power,MINBIO Power,MINCOA Power
Year,,,,,,,,,
2015,0,0,0,0,0,0,0,0,0
2016,0,0,0,0,0,0,0,0,0
2017,0,0,0,0,0,0,0,0,0
2018,0,0,0,0,0,0,0,0,0
2019,0,0,0,0,0,0,0,0,0


In [17]:
df_elec_production = net_zero_extrator.load(df_input_full,'elec_production')
df_elec_production.head(5)

,PWRBIO001,PWRCOA001,PWRCSP001,PWRCSP002,PWRDIST,PWRGEO,PWRHYD001,PWRHYD002,PWRHYD003,PWRHYD004,...,PWROHC004,PWRSOL001,PWRSOL001S,PWRSOL002,PWRSOL002S,PWRTRN,PWRTRNEXP,PWRTRNIMP,PWRWND001,PWRWND001S
Year,,,,,,,,,,,,,,,,,,,,,
2015,0.001,0,0,0,35.962576,0,21.2155,0,0,0,...,0,0.0152,0,0,0,41.66396,0,1.245,0,0
2016,23,0,0,0,39.386048,0,20.512,0,0,0,...,0,0.0968,0,0,0,45.519155,0,3.215,0,0
2017,0.0004,0,0,0,44.673572,0,20.652,0,0,0,...,0,0.152,0,0,0,51.504725,0,1.5642,0,0
2018,0.0004,0,0,0,49.122633,0,22.1,0,0,0,...,0,0.155,0,0,0,56.496975,0,0.7542,0,0
2019,0.0004,0,0,0,53.062384,0,26.950305,0,0,0,...,0,0.2531,0,0,0,60.880755,0,0.654,0,0


In [18]:
df_opex = net_zero_extrator.load(df_input_full,'opex')
df_opex.head(5)

,PWRBIO001,PWRCOA001,PWRCSP001,PWRCSP002,PWRDIST,PWRGEO,PWRHYD001,PWRHYD002,PWRHYD003,PWRHYD004,...,PWROHC004,PWRSOL001,PWRSOL001S,PWRSOL002,PWRSOL002S,PWRTRN,PWRTRNEXP,PWRTRNIMP,PWRWND001,PWRWND001S
Year,,,,,,,,,,,,,,,,,,,,,
2015,0.33,0,0,0,0.004166,0,142.562122,0,0,0,...,0,0.547454,0,0.169835,0,0.004386,0.000209,0.000124,0,0
2016,0.017738,0,0,0,0.004552,0,142.562051,0,0,0,...,0,0.447806,0,0.206228,0,0.004791,0.000268,0.000322,0,0
2017,0.017738,0,0,0,0.00515,0,142.562065,0,0,0,...,0,0.866849,0,0.23049,0,0.005422,0.000136,0.000156,0,0
2018,0.017738,0,0,0,0.00565,0,142.56221,0,0,0,...,0,1.317427,0,0.23049,0,0.005947,0.000266,0.000075,0,0
2019,0.017738,0,0,0,0.006088,0,142.562695,0,0,0,...,0,1.317437,0,0.242621,0,0.006409,0.000515,0.000065,0,0


In [19]:
df_potential_generation_nz = net_zero_extrator.load(df_input_full,'potential_generation')
df_potential_generation_nz.head(5)

,PWRBIO001,PWRCOA001,PWRCSP001,PWRCSP002,PWRDIST,PWRGEO,PWRHYD001,PWRHYD002,PWRHYD003,PWRHYD004,...,PWROHC004,PWRSOL001,PWRSOL001S,PWRSOL002,PWRSOL002S,PWRTRN,PWRTRNEXP,PWRTRNIMP,PWRWND001,PWRWND001S
Year,,,,,,,,,,,,,,,,,,,,,
2015,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2016,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2017,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2018,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2019,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


## Least Cost Scenario

In [20]:
least_cost_extrator = input_extractor('least_cost')
full_cap_cost_least_cost, df_ffe_lc,df_elec_production_lc = input_extractor.load_osemosys_input('least_cost',df_input_full)
df_capital_cost_lc = least_cost_extrator.load(df_input_full,'capital_cost')
df_capital_cost_lc.head(3)


,PWRBIO001,PWRCOA001,PWRCSP001,PWRCSP002,PWRDIST,PWRGEO,PWRHYD001,PWRHYD002,PWRHYD003,PWRHYD004,...,PWROHC004,PWRSOL001,PWRSOL001S,PWRSOL002,PWRSOL002S,PWRTRN,PWRTRNEXP,PWRTRNIMP,PWRWND001,PWRWND001S
Year,,,,,,,,,,,,,,,,,,,,,
2015,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2016,0,0,0,0,0,0,0,0,0,0,...,0,20.622568,0,0,0,0,0,0,0,0
2017,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [21]:
df_ffe_lc = least_cost_extrator.load(df_input_full,'ffe')
df_ffe_lc.head(3)


,IMPBIO Power,IMPCOA Power,IMPHFO Power,IMPLPG Power,IMPNGS Power,IMPOIL Power,IMPURN Power,MINBIO Power,MINCOA Power
Year,,,,,,,,,
2015,0,0,0,0,0,0,0,0,0
2016,0,0,0,0,0,0,0,0,0
2017,0,0,0,0,0,0,0,0,0


In [22]:
df_elec_production_lc = least_cost_extrator.load(df_input_full,'elec_production')
df_elec_production_lc.head(3)


,PWRBIO001,PWRCOA001,PWRCSP001,PWRCSP002,PWRDIST,PWRGEO,PWRHYD001,PWRHYD002,PWRHYD003,PWRHYD004,...,PWROHC004,PWRSOL001,PWRSOL001S,PWRSOL002,PWRSOL002S,PWRTRN,PWRTRNEXP,PWRTRNIMP,PWRWND001,PWRWND001S
Year,,,,,,,,,,,,,,,,,,,,,
2015,0.001,0,0,0,35.962576,0,21.2155,0,0,0,...,0,0.0152,0,0,0,41.66396,0,1.245,0,0
2016,0.0004,0,0,0,39.386048,0,20.512,0,0,0,...,0,0.0968,0,0,0,45.519155,0,3.215,0,0
2017,0.0004,0,0,0,44.673572,0,20.652,0,0,0,...,0,0.152,0,0,0,51.504725,0,1.5642,0,0


In [23]:
df_opex_lc = least_cost_extrator.load(df_input_full,'opex')
df_opex_lc.head(3)


,PWRBIO001,PWRCOA001,PWRCSP001,PWRCSP002,PWRDIST,PWRGEO,PWRHYD001,PWRHYD002,PWRHYD003,PWRHYD004,...,PWROHC004,PWRSOL001,PWRSOL001S,PWRSOL002,PWRSOL002S,PWRTRN,PWRTRNEXP,PWRTRNIMP,PWRWND001,PWRWND001S
Year,,,,,,,,,,,,,,,,,,,,,
2015,0.020762,0,0,0,0.004166,0,142.562122,0,0,0,...,0,0.547452,0,0.169835,0,0.004386,0.000209,0.000124,0,0
2016,0.017738,0,0,0,0.004552,0,142.562051,0,0,0,...,0,0.447796,0,0.206228,0,0.004791,0.000268,0.000322,0,0
2017,0.017738,0,0,0,0.00515,0,142.562065,0,0,0,...,0,0.866834,0,0.23049,0,0.005422,0.000136,0.000156,0,0


In [24]:
df_potential_generation_lc = least_cost_extrator.load(df_input_full,'potential_generation')
df_potential_generation_lc.head(3)

,PWRBIO001,PWRCOA001,PWRCSP001,PWRCSP002,PWRDIST,PWRGEO,PWRHYD001,PWRHYD002,PWRHYD003,PWRHYD004,...,PWROHC004,PWRSOL001,PWRSOL001S,PWRSOL002,PWRSOL002S,PWRTRN,PWRTRNEXP,PWRTRNIMP,PWRWND001,PWRWND001S
Year,,,,,,,,,,,,,,,,,,,,,
2015,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2016,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2017,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [25]:
least_cost_summary =  least_cost_extrator.get_totals(df_input_full)
least_cost_summary.head(5)

,variable_cost,fixed_cost,co2_emission,carbon_price,capital_cost,total_cost,annual_elec_production,cost_of_elec_in_pj,cost_of_elec,cost_of_co2
2025,0.03574,298.324291,53419.259,0,2907.56605,3205.926081,2280.036546,1.406085,0.390579,0.0
2026,0.035745,303.966973,54311.56,0,2399.29072,2703.293437,2281.167137,1.185048,0.32918,0.0
2027,0.036508,316.879256,55795.776,0,2667.41796,2984.333724,2289.467351,1.303506,0.362085,0.0
2028,0.037256,320.11022,57323.518,0,2284.21112,2604.358597,2297.629837,1.133498,0.314861,0.0
2029,0.038019,323.225433,58908.37,0,2280.91853,2604.181981,2305.930236,1.129341,0.313706,0.0


- We then calculate the CO2 savings from least cost and net zero scenarios


In [26]:
co2_savings = pd.DataFrame()
co2_savings['co2_savings'] = least_cost_summary['cost_of_co2']-net_zero_summary['cost_of_co2']
co2_savings.head(3)

,co2_savings
2025,0.0
2026,0.0
2027,0.0


- In the following cell, we calculate the total costs for different scenarios


In [27]:
#2025 -2070 period total
cost_keys = ['capital_cost','variable_cost','fixed_cost']
data = []
for key in cost_keys:
    data.append({
        "Scenarios": key,  # Scenarios: net zero / least cost
        "NetZero": net_zero_summary[key].sum(),# Million USD
        "LeastCost": least_cost_summary[key].sum(),#Million USD
    })
# net_zero_summary['capital_cost']
period_total = pd.DataFrame(data).T
period_total.columns = period_total.iloc[0] 
period_total = period_total[1:]
period_total['total'] = period_total.iloc[0:,:].sum(axis=1)
#Below is the data shown in the input sheet W137:AA138
period_total

Scenarios,capital_cost,variable_cost,fixed_cost,total
NetZero,1701826.235326,3329.83785,109406.796201,1814562.869377
LeastCost,141632.905312,3547.762929,22707.553382,167888.221623


### Slide window average 
- here we compute the 5 year windown average for various data.
- These are shown in the Excel Input sheet W141:AG151

In [28]:
five_year_intervals = list(range(2025, 2070, 5))  # Generates [2025, 2030, 2035, ..., 2065]
df_5yr_avg = pd.DataFrame({"Year": five_year_intervals})
# Compute 5-year averages for each interval
for key in cost_keys:
    df_5yr_avg[f"nz_{key}"] = net_zero_summary[key].groupby(pd.cut(net_zero_summary.index, 
                                                              bins=list(range(2025, 2071, 5)), 
                                                              labels=five_year_intervals,right=False),observed=False) \
                                             .mean().reset_index(drop=True)
    df_5yr_avg[f"lc_{key}"] = least_cost_summary[key].groupby(pd.cut(net_zero_summary.index, 
                                                              bins=list(range(2025, 2071, 5)), 
                                                              labels=five_year_intervals,right=False),observed=False) \
                                             .mean().reset_index(drop=True)
    df_5yr_avg[f"delta_{key}"] = df_5yr_avg[f"lc_{key}"] - df_5yr_avg[f"nz_{key}"]
    
df_5yr_avg["emission_saving"] = co2_savings["co2_savings"].groupby(pd.cut(net_zero_summary.index, 
                                                              bins=list(range(2025, 2071, 5)), 
                                                              labels=five_year_intervals,right=False),observed=False).mean().reset_index(drop=True)    
df_5yr_avg.head(3)     

,Year,nz_capital_cost,lc_capital_cost,delta_capital_cost,nz_variable_cost,lc_variable_cost,delta_variable_cost,nz_fixed_cost,lc_fixed_cost,delta_fixed_cost,emission_saving
0,2025,14270.945264,2507.880876,-11763.064388,0.04644,0.036653,-0.009787,472.218756,312.501235,-159.717521,0.0
1,2030,44183.050499,2914.373762,-41268.676738,0.062843,4.031509,3.968665,760.983336,332.998207,-427.985129,0.0
2,2035,52753.608356,3644.096076,-49109.51228,4.810418,45.260026,40.449608,1305.458182,398.410933,-907.047249,0.0


In [ ]:
import numpy as np
import pandas as pd
from MinFin import financing_baseline_extractor, financing_baseline_stats

df_financing_baseline_full = pd.read_excel(file_path, sheet_name="Financing Baseline")
df_financing_baseline_full
fb = financing_baseline_extractor(df_financing_baseline_full)

# Define the years from 2015 to 2079
years = np.arange(2000, 2080)

# Define some example currencies
currencies = ["EUR", "GBP", "JPY", "CNY", "INR", "AUD", "CAD"]

# Generate synthetic exchange rates with a simulated yearly change
np.random.seed(42)  # For reproducibility
base_rates = {
    'USD': 1#, "EUR": 1.1, "GBP": 1.3, "JPY": 110, "CNY": 6.5, "INR": 74, "AUD": 1.4, "CAD": 1.25
}
base_rates["KES"] = 1

# Simulate yearly fluctuations for KES (small changes)
fluctuations = np.cumsum(np.random.normal(0, 0.01, len(years)))  # Simulated yearly change
# Create a DataFrame to store the exchange rates
exchange_rates = pd.DataFrame({"Year": years})
exchange_rates["KES"] = base_rates["KES"]# * (1 + fluctuations)
# exchange_rates["USD"] = base_rates["USD"]
# Simulate yearly fluctuations in exchange rates
for currency, base_rate in base_rates.items():
    fluctuations = np.cumsum(np.random.normal(0, 0.01, len(years)))  # Simulated yearly change
    exchange_rates[currency] = base_rate #* (1 + fluctuations)
exchange_rates = fb.get_exchange_rates_by_year()
melted_currency = exchange_rates.reset_index().melt(id_vars=["index"], var_name="Currency", value_name="Exchange Rate")
# fb.get_exchange_rates_by_year()
melted_currency
exchange_rates

## Funding Baseline Sheet

Here we read and process the historic funding information from the excel sheet

``Retuires user input``

In [ ]:
# Load the "Funding Baseline" sheet from the Excel file

from MinFin import process_funding_baseline

# Read the full "Funding Baseline" sheet
df_funding_baseline_full = pd.read_excel(file_path, sheet_name="Funding Baseline", engine="openpyxl")
df_final = process_funding_baseline(df_funding_baseline_full,melted_currency)
df_funding_baseline=process_funding_baseline(df_funding_baseline_full,melted_currency)
df_funding_baseline.iloc[50:]
df_final.head(10)

c:\Users\Zixuan\AppData\Local\Programs\Python\Python312\Lib\site-packages\openpyxl\worksheet\_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)


,Source,Name,Type,Year,Volume (Million),Currency,Govt Share,Volume (Foreign Currency USD),level_0,Exchange Rate,volume_in_usd
0,GoG MoE Budget,GoG MoG Budget,Budget,2024,74.39,USD,1,74.39,14.0,1,74.39
1,GoG MoE Budget,GoG MoG Budget,Budget,2023,40.59,USD,1,40.59,13.0,1,40.59
2,GoG MoE Budget,GoG MoG Budget,Budget,2022,19.98,USD,1,19.98,12.0,1,19.98
3,GoG MoE Budget,GoG MoG Budget,Budget,2021,126.46,USD,1,126.46,11.0,1,126.46
4,GoG MoE Budget,GoG MoG Budget,Budget,2020,88.68,USD,1,88.68,10.0,1,88.68
5,GoG MoE Budget,GoG MoG Budget,Budget,2019,78.8,USD,1,78.8,9.0,1,78.8
6,GoG MoE Budget,GoG MoG Budget,Budget,2018,22.89,USD,1,22.89,8.0,1,22.89
7,GoG MoE Budget,GoG MoG Budget,Budget,2017,126,USD,1,126,7.0,1,126.0
8,GoG MoE Budget,GoG MoG Budget,Budget,2016,224.55,USD,1,224.55,6.0,1,224.55
9,GoG MoE Budget,GoG MoG Budget,Budget,2015,135.13,USD,1,135.13,5.0,1,135.13


- And get the funding envolope from the input
  
> This is A5:R8 in the Funding Baseline sheet
>

*``Note that Budget statistics are to be determined in Excel, same calculation used here for data verification``*

`` Question to Team ``

> In LogReg mode, any negative values are ignores while others are not.

In [ ]:
from MinFin import get_funding_envelope

df_funding_envelope=get_funding_envelope(df_funding_baseline)
# 只对 index 名字为年份（数字）的行计算平均值，再添加为"Annual Average"
df_funding_envelope

Type,Budget,Grant,SOE Gen.
Year,,,
2010,163.53,42.64,11.606
2011,166.3,27.51,39.491
2012,225.51,29.45,34.942
2013,327.31,11.58,11.602
2014,285.8,9.36,226.389
2015,135.13,12.57,145.772
2016,224.55,5.32,224.793
2017,126.0,4.99,135.694
2018,22.89,3.47,42.569


## Investment Needs 
- with input we have processed, we now calculate the investment needs
> Investment Needs is updated to a newer version, where names/categories are both from the definition sheet.
> FFRM related calculations removed

In [ ]:
import pandas as pd
import numpy as np

global since_year 
since_year= 2025
def filter_data(data):
    # 如果没有year列 就用index
    col_to_use = "Year" if "Year" in data.columns else None
    if col_to_use is None:
        # Use index for year filtering
        data_index_numeric = pd.to_numeric(data.index, errors="coerce")
        filtered = data.loc[data_index_numeric >= since_year]
        return filtered
    else: 
        data["Year"] = pd.to_numeric(data["Year"], errors='coerce')
        data = data[data["Year"]>=since_year]
    return data

def aggregate_by_mapping(df, mapping):
    """
    Aggregate data by col names
    df: 
    mapping: dict {name -> classification} or {name -> technology}
    """
    tmp = df.rename(columns=mapping)          # map column name to desired groups
    tmp = tmp.T.groupby(level=0).sum().T      # Sum up cols with same names
    return filter_data(tmp)                   # filter data by year
def get_tech_cls_map(technologies):
    return {tech.name: tech.classification for tech in technologies}
    

def cal_invest_needs(technologies,least_cost_summary,net_zero_summary,df_ffe_least_cost,df_ffe_net_zero,full_cap_cost_least_cost,full_cap_cost_net_zero):
    """Extracts and constructs the Investment Needs table from OSeMOSYS and FFRM sheets."""
    df_emission_savings = pd.DataFrame()
    df_emission_savings['emission_savings'] = least_cost_summary['co2_emission']-net_zero_summary['co2_emission']
    
    '''
    Savings on Fossil Fuel Expenditure
    '''
    fossil_fuel_savings = pd.DataFrame()
    
    fossil_fuel_savings = filter_data(df_ffe_least_cost)-filter_data(df_ffe_net_zero)
    fossil_fuel_no_earnings = fossil_fuel_savings.drop(columns=["Total", "Year"], errors="ignore").copy()  # Assuming df_funding_develop is the DataFrame

    fossil_fuel_no_earnings.iloc[:, :] = fossil_fuel_no_earnings.map(lambda x: max(x, 0))
    # Year may be index or cols
    filtered_least_cost = filter_data(df_ffe_least_cost)
    if "Year" in filtered_least_cost.columns:
        fossil_fuel_savings['Year'] = filtered_least_cost["Year"].astype(int)
    else:
        fossil_fuel_savings['Year'] = pd.to_numeric(filtered_least_cost.index, errors="coerce").astype(int)
    fossil_fuel_savings["expenditure"] = fossil_fuel_no_earnings.sum(axis=1)
    fossil_fuel_savings=fossil_fuel_savings.set_index("Year")
    # for key in ["coal","oil","gas","biomass"]:
    #     fossil_fuel_savings[key] = 
    '''
    Financing Needs for FFR （FFRM is temporarily removed in MinFin, they are now set as 0）
    '''

    tech_list = ['Oil', 'Gas', 'Coal']
    total_cols = [(tech, tech) for tech in tech_list]

    # 生成列：(Oil, Oil), (Gas, Gas), (Coal, Coal), (Total, Total)
    cols = total_cols + [('Total', 'Total')]

    # 按 fossil_fuel_savings 行数建表，全部填 0
    n = len(fossil_fuel_savings.index)
    df_financing_needs_ffr = pd.DataFrame(
        0.0,
        index=fossil_fuel_savings.index,
        columns=pd.MultiIndex.from_tuples(cols),
    )
    df_financing_needs_ffr.insert(0, ("General", "Year"), df_financing_needs_ffr.index)

    # Total 列 = 三列之和（当前为 0）
    df_financing_needs_ffr.loc[:, ('Total', 'Total')] = df_financing_needs_ffr.loc[:, total_cols].sum(axis=1)

    '''
    Least Cost Financing needs
    '''
    tech_category_map = get_tech_cls_map(technologies)

    # name -> Technology（df_technologies）
    name_to_tech_map = df_technologies.set_index("Name")["Technology"].to_dict()

    #### LEAST COST：classification ####
    df_cap_by_class_lc = aggregate_by_mapping(full_cap_cost_least_cost, tech_category_map)

    #### LEAST COST:Technology ####
    df_cap_by_tech_lc = aggregate_by_mapping(full_cap_cost_least_cost, name_to_tech_map)
    #### NET ZERO： classification ####
    df_cap_by_class_nz = aggregate_by_mapping(full_cap_cost_net_zero, tech_category_map)

    #### NET ZERO：Technology ####
    df_cap_by_tech_nz = aggregate_by_mapping(full_cap_cost_net_zero, name_to_tech_map)
    
    # Least Cost：class + tech 
    df_category_sum_lc = pd.concat(
        [df_cap_by_class_lc, df_cap_by_tech_lc],
        axis=1
    )

    # Net Zero：class + tech 
    df_category_sum_nz = pd.concat(
        [df_cap_by_class_nz, df_cap_by_tech_nz],
        axis=1
    )
    return df_cap_by_class_lc, df_cap_by_tech_lc,df_cap_by_class_nz, df_cap_by_tech_nz,df_emission_savings,df_financing_needs_ffr.set_index(("General","Year"),drop=False),fossil_fuel_savings

# Example Usage

df_cap_by_class_lc, df_cap_by_tech_lc,df_cap_by_class_nz, df_cap_by_tech_nz,df_emission_savings,df_financing_needs_ffr,fossil_fuel_savings= cal_invest_needs(technologies,least_cost_summary,net_zero_summary,df_ffe_lc,df_ffe,full_cap_cost_least_cost,full_cap_cost_net_zero)
# fossil_fuel_savings                         

In [ ]:
    # Least Cost：class + tech 
    df_category_sum_lc = pd.concat(
        [df_cap_by_class_lc, df_cap_by_tech_lc],
        axis=1
    )

    # Net Zero：class + tech 
    df_category_sum_nz = pd.concat(
        [df_cap_by_class_nz, df_cap_by_tech_nz],
        axis=1
    )

In [ ]:
def get_category_sum(by_class,by_tech):
    """
    Concatenate by_class and by_tech dataframes along columns
    """
    return pd.concat(
        [by_class, by_tech],
        axis=1
    )

df_category_sum_lc = get_category_sum(df_cap_by_class_lc, df_cap_by_tech_lc)
df_category_sum_nz = get_category_sum(df_cap_by_class_nz, df_cap_by_tech_nz)
df_category_sum_nz.head(3)

,Distribution Infrastructure,Exports,Generation Fossil-Fuel,Generation Renewable,Transmission Infrastructure,Biomass,CSP,Coal,Direct Hydro,Direct Oil,...,Energy Exports,Gas,Geothermal,Hydropower,Imports,Nuclear,Oil,Solar PV,Transmission,Wind
Year,,,,,,,,,,,,,,,,,,,,,
2025,447.12062,0,1993.33772,4058.520861,0,56,192,96,467.21633,948.66886,...,0,0,447.12062,1068.69277,0.000001,0,948.66886,913.74557,0,0
2026,1631.421,913.74557,2141.18721,4900.593582,913.74557,46,212,106,474.33443,963.3464,...,913.74557,108.49441,203.22078,547.19878,0.000002,0,963.3464,1252.04701,913.74557,913.74557
2027,2781.8555,338.30144,2336.925202,3740.279736,338.30144,36,0.106144,0.053072,481.6732,978.47894,...,338.30144,379.91425,447.12062,801.76193,0.000002,0,978.47894,817.6582,338.30144,338.30144


In [ ]:
df_category_sum_nz.head(3)

,Distribution Infrastructure,Exports,Generation Fossil-Fuel,Generation Renewable,Transmission Infrastructure,Biomass,CSP,Coal,Direct Hydro,Direct Oil,...,Energy Exports,Gas,Geothermal,Hydropower,Imports,Nuclear,Oil,Solar PV,Transmission,Wind
Year,,,,,,,,,,,,,,,,,,,,,
2025,447.12062,0,1993.33772,4058.520861,0,56,192,96,467.21633,948.66886,...,0,0,447.12062,1068.69277,0.000001,0,948.66886,913.74557,0,0
2026,1631.421,913.74557,2141.18721,4900.593582,913.74557,46,212,106,474.33443,963.3464,...,913.74557,108.49441,203.22078,547.19878,0.000002,0,963.3464,1252.04701,913.74557,913.74557
2027,2781.8555,338.30144,2336.925202,3740.279736,338.30144,36,0.106144,0.053072,481.6732,978.47894,...,338.30144,379.91425,447.12062,801.76193,0.000002,0,978.47894,817.6582,338.30144,338.30144


In [ ]:
df_emission_savings.T

,2025,2026,2027,2028,2029,2030,2031,2032,2033,2034,...,2061,2062,2063,2064,2065,2066,2067,2068,2069,2070
emission_savings,7600.571,7742.594,8942.207,11023.187,13232.516,16552.537,19720.662,22787.873,25635.945,28399.456,...,107406.9,109261.31,111168.86,113131.06,115149.43,117225.52,119360.88,121764.57,124240.13,126791.47


In [ ]:
fossil_fuel_savings.T

Year,2025,2026,2027,2028,2029,2030,2031,2032,2033,2034,...,2061,2062,2063,2064,2065,2066,2067,2068,2069,2070
IMPBIO Power,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
IMPCOA Power,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
IMPHFO Power,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
IMPLPG Power,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
IMPNGS Power,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
IMPOIL Power,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
IMPURN Power,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
MINBIO Power,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
MINCOA Power,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
expenditure,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [ ]:

df_category_sum_incremental = df_category_sum_nz-df_category_sum_lc 
df_category_sum_incremental.head(3)



,Distribution Infrastructure,Exports,Generation Fossil-Fuel,Generation Renewable,Transmission Infrastructure,Biomass,CSP,Coal,Direct Hydro,Direct Oil,...,Energy Exports,Gas,Geothermal,Hydropower,Imports,Nuclear,Oil,Solar PV,Transmission,Wind
Year,,,,,,,,,,,,,,,,,,,,,
2025,447.12062,0,1993.33772,3175.954811,0,56,192,96,467.21633,948.66886,...,0,0,447.12062,1068.69277,0.000001,0,948.66886,31.17952,0,0
2026,1631.421,913.74557,2032.6928,4635.797272,913.74557,46,212,106,474.33443,963.3464,...,913.74557,0.0,203.22078,547.19878,0.000002,0,963.3464,987.2507,913.74557,913.74557
2027,2781.8555,338.30144,1957.010952,3479.776026,338.30144,36,0.106144,0.053072,481.6732,978.47894,...,338.30144,0.0,447.12062,801.76193,0.000002,0,978.47894,557.15449,338.30144,338.30144


In [ ]:
total_financing_needs_nz = df_category_sum_nz.sum(axis=1)/2
total_financing_needs_lc = df_category_sum_lc.sum(axis=1)/2
total_financing_needs_lc_incremental = df_category_sum_incremental.sum(axis=1)/2
df_invest_need_summary = pd.DataFrame({
    'Net Zero': total_financing_needs_nz,
    'Least Cost': total_financing_needs_lc,
    'Incremental': total_financing_needs_lc_incremental
})
df_invest_need_summary.loc[:, "Total financing"] = df_invest_need_summary.loc[:, "Net Zero"]

df_invest_need_summary


,Net Zero,Least Cost,Incremental,Total financing
Year,,,,
2025,6498.979201,882.56605,5616.413151,6498.979201
2026,10500.692932,373.29072,10127.402212,10500.692932
2027,9535.663318,640.41796,8895.245358,9535.663318
2028,12333.219742,256.21112,12077.008622,12333.219742
2029,22531.171126,251.91853,22279.252596,22531.171126
2030,42086.07542,247.62594,41838.44948,42086.07542
2031,40750.626594,578.022367,40172.604227,40750.626594
2032,30966.418357,754.489211,30211.929146,30966.418357
2033,37908.254391,1144.10409,36764.150301,37908.254391


## Financing Baseline
``Retuires user input``
Modalities of Financing – Grants, Debt and Equity

`Grants`:
- The most concessional source of financing (and typically smallest)
- This is “free money” that requires no financial repayments.

`Debts`:
- This is typically the largest source of project financing
- Repayments are dependent on Loan Structure, Interest Rate, Grace period and Loan Term

`Equity`:
- This mode of  finance is the highest risk, and expects highest returns
- Repayments are based on the revenues generated by the project


In the following cell, we import some functions from the MinFin module for Financning baseline to extract historic financing data and calculate repayment schedule and its statistics:

In [ ]:
import pandas as pd
from MinFin import financing_baseline_extractor, financing_baseline_stats

df_financing_baseline_full = pd.read_excel(file_path, sheet_name="Financing Baseline")
df_financing_baseline_full
fb = financing_baseline_extractor(df_financing_baseline_full)
historical = fb.get_historical()
# # historical
repayment_schedule = fb.cal_repayment_schedule(historical)
fbs= financing_baseline_stats(fb,repayment_schedule)
repayment_statistics= fbs.get_repayment_statistics()


c:\Users\Zixuan\AppData\Local\Programs\Python\Python312\Lib\site-packages\openpyxl\worksheet\_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)


interest_rate 0.0075
volume 70
start_year 2010
term 40
grace_period 10
year 2010
interest_rate 0.0075
volume 70
start_year 2010
term 40
grace_period 10
year 2011
interest_rate 0.0251
volume 75.8
start_year 2011
term 14.26
grace_period 2.76
year 2011
interest_rate 0.0806
volume 100
start_year 2011
term 20
grace_period 4
year 2011
interest_rate 0.0565
volume 100
start_year 2011
term 20
grace_period 4
year 2011
interest_rate 0.02
volume 75.4
start_year 2012
term 20
grace_period 5
year 2012
interest_rate 0
volume 76.2
start_year 2012
term 14
grace_period 2
year 2012
interest_rate 0.053
volume 210
start_year 2012
term 15
grace_period 5
year 2012
interest_rate 0.053
volume 120
start_year 2012
term 15
grace_period 5
year 2012
interest_rate 0.0149
volume 0
start_year 2012
term 15
grace_period 5
year 2012
interest_rate 0.0565
volume 9
start_year 2012
term 11
grace_period 0
year 2012
interest_rate 0.0025
volume 64.6
start_year 2012
term 20
grace_period 8
year 2012
interest_rate 0.0099
volume 102

c:\Users\Zixuan\Dropbox\CCG\MinFin\MinFin\financing_baseline.py:375: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  debt_share = repayment_schedule[
c:\Users\Zixuan\Dropbox\CCG\MinFin\MinFin\financing_baseline.py:375: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  debt_share = repayment_schedule[
c:\Users\Zixuan\Dropbox\CCG\MinFin\MinFin\financing_baseline.py:375: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  debt_share = repayment_schedule[
c:\Users\Zixuan\Dropbox\CCG\MinFin\MinFin\financing_baseline.py:375: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  debt_share = repayment_schedule[


## Here we extract the currency/exchange rate information: 
- We currenly generate random rates, can extract from the internet or set to be user-defined.
- It is not taking effect in the further processing of the data, as we use USD as default

## Repaying the finance

- Repayments differ based on the Type of Finance and Repayment Schedule:

| Year | Annuity       |                  | Annuity + 3yr Grace |                  | EPP           |              | EPP + 3yr Grace on Principal Only |       | EPP + 3yr Grace on P & I |       | Lump Sum (Principal) |       | Lump Sum (P + Compound I) |       |
|------|----------------|------------------|----------------------|------------------|----------------|--------------|-----------------------------------|-------|----------------------------|-------|------------------------|-------|----------------------------|-------|
|      | Principal (P)  | Interest (I)     |  (P)        |  (I)     |  (P)  |  (I) |  (P)                     | (I)   |  (P)             | (I)   |  (P)          | (I)   |  (P)             | (I)   |
| 1    | ●●            | ●              |                      |                  | ●●            | ●●●          |                                  | ●●   |                            |    |                        | ●●   |                            |       |
| 2    | ●●            | ●              |                      |                  | ●●            | ●●          |                                  | ●●   |                            |    |                        | ●●   |                            |       |
| 3    | ●●            | ●              | ●●                 | ●              | ●●           | ●●          | ●●                              | ●●   | ●●                           | ●●   |                        | ●●   |                            |       |
| 4    | ●●            | ●              | ●●                  | ●              | ●●            | ●          | ●●                              | ●●   | ●●                        | ●●   |                        |●●       |                            |       |
| 5    | ●●            | ●              | ●●                  | ●             | ●●            | ●          | ●●                              | ●  | ●●                        | ●   | ●●●                    | ●●      | 
●●●                        | ●●●   |

In the following cell we show the repayment schedule,


In [ ]:
historical

,Name of Project,Name of Financier,Sector,Technology,Source,Year,Financing Source,Type of Finance,Financing Sector,Financial Institution,...,Volume in GHS,Rate,Term,Grace period,Schedule,Volume in KES,Volume in USD,Exchange Rate to KES,Exchange Rate to USD,Maturity
0,Bui Dam Power Plant (Additional Loan),China Ex-im Bank,NaN,Hydropower,Renewable,2012,Conc_IFI,Loan,Public,Bilateral Agency,...,137.228,0.02,20,5,EPP with Grace on Principal & Interest,50.266667,75.4,0.666667,1.0,8
1,Bui Dam Power Plant (Additional Loan),China Ex-im Bank,NaN,Hydropower,Renewable,2012,Conc_IFI,Loan,Public,Bilateral Agency,...,138.684,0,14,2,EPP with Grace on Principal & Interest,50.8,76.2,0.666667,1.0,2
2,Kpone Independent Power Project (CCGT),Nederlandse Financierings-Maatschappij voor On...,NaN,Gas,Fossil Fuel,2014,Conc_IFI,Loan,Public,Bilateral Agency,...,215.47,0.015,20,4,EPP with Grace on Principal & Interest,49.533333,74.3,0.666667,1.0,10
3,Kpone Independent Power Project (CCGT),Deutsche Investitions-und Entwicklungsgesellsc...,NaN,Gas,Fossil Fuel,2014,Conc_IFI,Loan,Public,Bilateral Agency,...,215.47,0.015,20,4,EPP with Grace on Principal & Interest,49.533333,74.3,0.666667,1.0,10
4,Kpone Independent Power Project (CCGT),OPEC Fund for International Development (OFID),NaN,Gas,Fossil Fuel,2014,Conc_IFI,Loan,Public,Bilateral Agency,...,215.47,0.015,20,4,EPP with Grace on Principal & Interest,49.533333,74.3,0.666667,1.0,10
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
62,Ameri Aboadze Power Plant,Africa and Middle East Resource Investment,NaN,Gas,Fossil Fuel,2014,Comm_Intl,Loan,Public,Private Equity Fund,...,1479,0.225,20,0,Equity,340.0,510.0,0.666667,1.0,10
63,Karpowership Power Plant,Karpowership Ghana Company Limited,NaN,Gas,Fossil Fuel,2014,Comm_Intl,Equity,Private,Private Equity Fund,...,1740,0.225,20,0,Equity,400.0,600.0,0.666667,1.0,10
64,Aksa Enerji Oil Fired Power Plant,Aksa Enerji Uretim AS,NaN,Oil,Fossil Fuel,2016,Comm_Intl,Equity,Private,Private Equity Fund,...,0,0.225,25,0,Equity,NaN,NaN,0.666667,1.0,17
65,Bui Solar Power Plant,Absa Bank (Ghana),NaN,Solar PV,Renewable,2023,Comm_Intl,Loan,Private,Commercial Bank,...,0,0.0888,14,0,EPP with Grace on Principal & Interest,NaN,NaN,0.666667,1.0,13


>Discussion Point: Currency coversion

In [ ]:
repayment_schedule.T.head(50)


,0,1,2,3,4,5,6,7,8,9,...,57,58,59,60,61,62,63,64,65,66
2010,0,0,0,0,0,0,0,0,0,0,...,0.0,0,0,0,0,0,0,0,0,0
2011,0,0,0,0,0,0,0,0,0,0,...,0.0,0.0,0,0.0,0.0,0,0,0,0,0
2012,0.0,0.0,0,0,0,0,0,0,0,0,...,0.0,0.0,0,0.0,0.0,0,0,0,0,0
2013,0.0,0.0,0,0,0,0,0,0,0,0,...,0.0,3.66161,0,0.0,0.0,0,0,0,0,0
2014,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.8444,2.8444,...,0.0,8.883904,0,0.0,0.0,114.75,135.0,0,0,0
2015,0.0,6.35,0.0,0.0,0.0,0.0,0.0,0.0,5.43718,5.43718,...,0.0,8.710053,0,10.989929,7.039251,114.75,135.0,0,0,0
2016,0.0,6.35,0.0,0.0,0.0,0.0,0.0,0.0,5.29496,5.29496,...,0.0,8.536202,0.0,18.825026,14.38608,114.75,135.0,NaN,0,0
2017,1.664954,6.35,0.0,0.0,0.0,0.0,0.0,0.0,5.15274,5.15274,...,0.0,8.362351,0.0,18.138155,13.946127,114.75,135.0,NaN,0,28.919083
2018,7.103803,6.35,1.18289,1.18289,1.18289,1.18289,1.18289,1.18289,5.01052,5.01052,...,0.0,8.1885,0.0,17.451285,13.506174,114.75,135.0,NaN,0,55.261607
2019,6.992806,6.35,6.037666,6.037666,6.037666,6.037666,6.037666,6.037666,4.8683,4.8683,...,0.0,8.014649,0.0,16.764414,13.06622,114.75,135.0,NaN,0,53.98723


In [ ]:
repayment_statistics

Volume (USD) Interest rate       Term  \
Summary Total financing volumes   5181.69874       0.09848  20.607191   
        Conc_IFI                  2175.51874      0.023848   22.55331   
        Conc_DPS                      576.96      0.043495  13.663512   
        Comm_Intl                    2145.32      0.173584  20.157509   
        Comm_Dom                       283.9      0.214591  23.203593   
Equity  Total financing volumes       1383.9      0.212864  21.014885   
        Conc_IFI                        90.0      0.071222       20.0   
        Conc_DPS                           0             0          0   
        Comm_Intl                     1010.0         0.225  20.490099   
        Comm_Dom                       283.9      0.214591  23.203593   
Debt    Total financing volumes   3797.79874      0.056799   20.45863   
        Conc_IFI                  2085.51874      0.021804  22.663498   
        Conc_DPS                      576.96      0.043495  13.663512   
        Comm_Intl                    1135.32      0.127843  19.861632   
        Comm_Dom                           0             0          0   

                                Grace period Average Annual Payment  \
Summary Total financing volumes     2.222273              37.440542   
        Conc_IFI                    4.614259                7.01444   
        Conc_DPS                     2.14596               32.50375   
        Comm_Intl                   0.111223              73.221803   
        Comm_Dom                         0.0              10.243041   
Equity  Total financing volumes          0.0              69.488365   
        Conc_IFI                         0.0               4.145167   
        Conc_DPS                           0                      0   
        Comm_Intl                        0.0              91.964242   
        Comm_Dom                         0.0              10.243041   
Debt    Total financing volumes     3.032059              25.762466   
        Conc_IFI                    4.813386               7.138263   
        Conc_DPS                     2.14596               32.50375   
        Comm_Intl                   0.210168              56.548211   
        Comm_Dom                           0                      0   

                                 Debt Share  Equity Share  Market Element  \
Summary Total financing volumes    0.732925      0.267075             NaN   
        Conc_IFI                   0.958631      0.041369             NaN   
        Conc_DPS                   1.000000      0.000000             NaN   
        Comm_Intl                  0.529208      0.470792             NaN   
        Comm_Dom                   0.000000      1.000000             NaN   
Equity  Total financing volumes         NaN           NaN             NaN   
        Conc_IFI                        NaN           NaN             NaN   
        Conc_DPS                        NaN           NaN             NaN   
        Comm_Intl                       NaN           NaN             NaN   
        Comm_Dom                        NaN           NaN             NaN   
Debt    Total financing volumes         NaN           NaN        0.226792   
        Conc_IFI                        NaN           NaN        0.180929   
        Conc_DPS                        NaN           NaN        0.428760   
        Comm_Intl                       NaN           NaN        0.208403   
        Comm_Dom                        NaN           NaN        0.000000   

                                 Grant Element  
Summary Total financing volumes            NaN  
        Conc_IFI                           NaN  
        Conc_DPS                           NaN  
        Comm_Intl                          NaN  
        Comm_Dom                           NaN  
Equity  Total financing volumes            NaN  
        Conc_IFI                           NaN  
        Conc_DPS                           NaN  
        Comm_Intl                          NaN  
   

In [ ]:
institution_shares = fbs.get_institution_shares()
institution_shares

,Share,Market Element,Debt Share,Equity Share
Bilateral Agency,0.297115,0.168279,0.993505,0.006495
Multilateral Agency,0.090665,0.110880,1.000000,0.000000
Foreign Government,0.002567,0.078539,1.000000,0.000000
National Government,0.111346,0.349317,1.000000,0.000000
Domestic Public Sector,0.000000,NaN,NaN,NaN
Climate Funds,0.000000,NaN,NaN,NaN
Commercial Bank,0.150179,0.390442,0.897196,0.102804
Private Equity Fund,0.348129,NaN,0.000000,1.000000


In [ ]:
financing_sector_shares = fbs.get_financing_sector_shares()

financing_sector_shares


,Share
Public,0.600116
Private,0.399884
Domestic,0.166135
International,0.833865


In [ ]:
# 读取“Technology Disag（S1）”sheet里面的内容
technology_disag_s1 = pd.read_excel(file_path, sheet_name='Technology Disag (S1)')
technology_disag_s1.head()


c:\Users\Zixuan\AppData\Local\Programs\Python\Python312\Lib\site-packages\openpyxl\worksheet\_reader.py:329: UserWarning: Conditional Formatting extension is not supported and will be removed
  warn(msg)
c:\Users\Zixuan\AppData\Local\Programs\Python\Python312\Lib\site-packages\openpyxl\worksheet\_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)


,MINFin,Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,...,Unnamed: 87,Unnamed: 88,Unnamed: 89,Unnamed: 90,Unnamed: 91,Unnamed: 92,Unnamed: 93,Unnamed: 94,Unnamed: 95,Unnamed: 96
0,Net Zero Financing Requirements,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,Comm_Intl,Comm_Intl,Comm_Intl,Comm_Intl,Comm_Intl,Comm_Intl,Comm_Intl,...,Conc_DPS,Conc_DPS,Conc_DPS,Conc_DPS,Conc_DPS,Conc_DPS,Conc_DPS,Conc_DPS,NaN,NaN
2,NaN,Technologies,WACC,Comm_Intl,NaN,NaN,NaN,NaN,NaN,NaN,...,Conc_DPS,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,Debt,NaN,NaN,Equity,NaN,Financing Shares,NaN,...,Debt,NaN,NaN,Equity,NaN,Financing Shares,NaN,NaN,Foreign Currency Shares,NaN
4,NaN,Variable,NaN,Interest Rate,Grace Period,Loan Term,Rate of Return,Project Life,Debt Share,Equity Share,...,Interest Rate,Grace Period,Loan Term,Rate of Return,Project Life,Debt Share,Equity Share,Share of Finance,Debt,Equity


In [ ]:
# 提取图片中显示的那部分数据（三个主要部分：默认、Comm_Dom、Conc_IFI）
# 先读取原始数据，不设置表头，以便处理多级表头
technology_disag_s1_raw = pd.read_excel(file_path, sheet_name='Technology Disag (S1)', header=None)

start_col = 3  # B列
end_col = 43   # AQ列（索引30，所以end_col=31）
start_row = 3
end_row = 61   # 表头3行+数据8行

technology_disag_s1_extracted = technology_disag_s1_raw.iloc[start_row:end_row, start_col:end_col].copy()

# 设置多级表头（前3行作为表头）
header_level1 = technology_disag_s1_extracted.iloc[0, :].values
header_level2 = technology_disag_s1_extracted.iloc[1, :].values
header_level3 = technology_disag_s1_extracted.iloc[2, :].values

# 清理 NaN 值：用前一个非空值填充，或者用空字符串
header_level1 = pd.Series(header_level1).fillna(method='ffill').fillna('').values
header_level2 = pd.Series(header_level2).fillna(method='ffill').fillna('').values
header_level3 = pd.Series(header_level3).fillna('').values  # 最后一层通常不需要前向填充

# 创建多级列名
technology_disag_s1_extracted.columns = pd.MultiIndex.from_arrays([
    header_level1,
    header_level2,
    header_level3
], names=['Source', 'Category', 'Parameter'])

technology_disag_s1_data = technology_disag_s1_extracted.iloc[3:].reset_index(drop=True)

c:\Users\Zixuan\AppData\Local\Programs\Python\Python312\Lib\site-packages\openpyxl\worksheet\_reader.py:329: UserWarning: Conditional Formatting extension is not supported and will be removed
  warn(msg)
c:\Users\Zixuan\AppData\Local\Programs\Python\Python312\Lib\site-packages\openpyxl\worksheet\_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)
C:\Users\Zixuan\AppData\Local\Temp\ipykernel_18348\2417403156.py:18: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  header_level1 = pd.Series(header_level1).fillna(method='ffill').fillna('').values
C:\Users\Zixuan\AppData\Local\Temp\ipykernel_18348\2417403156.py:19: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  header_level2 = pd.Series(header_level2).fillna(method='ffill').fillna('').values


In [ ]:
technology_disag_s1_data.head()

Source        Comm_Intl                                                     \
Category           Debt                                Equity                
Parameter Interest Rate Grace Period Loan Term Rate of Return Project Life   
0                   NaN          NaN       NaN            NaN          NaN   
1              0.269247         0.21     19.74       0.284547           30   
2                   NaN          NaN       NaN            NaN          NaN   
3                   NaN          NaN       NaN            NaN          NaN   
4              0.265747         0.21     19.74       0.281047           30   

Source                                                    \
Category  Financing Shares                                 
Parameter       Debt Share Equity Share Share of Finance   
0                      NaN          NaN              NaN   
1                   0.5344       0.4656              0.7   
2                      NaN          NaN              NaN   
3                      NaN          NaN              NaN   
4                   0.5962       0.4038              0.7   

Source                                    ...      Conc_DPS               \
Category  Foreign Currency Shares         ...          Debt                
Parameter                    Debt Equity  ... Interest Rate Grace Period   
0                             NaN    NaN  ...           NaN          NaN   
1                               1      1  ...        0.0435         2.15   
2                             NaN    NaN  ...           NaN          NaN   
3                             NaN    NaN  ...           NaN          NaN   
4                               1      1  ...      0.027527     0.583974   

Source                                                                         \
Category                    Equity              Financing Shares                
Parameter Loan Term Rate of Return Project Life       Debt Share Equity Share   
0               NaN            NaN          NaN              NaN          NaN   
1             13.66            NaN          NaN                1            0   
2               NaN            NaN          NaN              NaN          NaN   
3               NaN            NaN          NaN              NaN          NaN   
4          8.013943            NaN           50                1            0   

Source                                                     
Category                   Foreign Currency Shares         
Parameter Share of Finance                    Debt Equity  
0                      NaN                     NaN    NaN  
1                   0.1108                       0      0  
2                      NaN                     NaN    NaN  
3                      NaN                     NaN    NaN  
4                    0.111                       1    NaN  

[5 rows x 40 columns]

> In the following cell, we match the order of the technologies as they appear in the excel sheet, without linking them to the cells, but linking them to previously defined variables.

In [ ]:

tech_order = {}
for idx, row in df_technologies.iterrows():
    tech_name = row['Technology']
    if tech_name not in tech_order:
        tech_order[tech_name] = idx  # 记录第一次出现的索引
        
# 然后按 class 分组，每个 class 后面跟着属于它的 tech
ordered_index = []
for class_name in df_technologies_classification['Classification'].unique():
    # 先添加 class
    ordered_index.append(class_name)
    
    # 获取属于这个 class 的所有 tech
    class_techs = []
    for tech in df_technologies_classification['Technology']:
        if tech_to_class_map.get(tech, '') == class_name:
            class_techs.append(tech)
    
        # 按照在 df_technologies 中第一次出现的顺序排序
    class_techs_sorted = sorted(
        class_techs,
        key=lambda x: tech_order.get(x, float('inf'))  # 如果找不到，放到最后
    )
    ordered_index.extend(class_techs_sorted)

    # 添加到 ordered_index*
print(len(ordered_index))
technology_disag_s1_data = technology_disag_s1_data.fillna(0).iloc[:len(ordered_index)]
technology_disag_s1_data.index = ordered_index
technology_disag_s1_data.T.head(40)



22


Generation Renewable  \
Source    Category                Parameter                               
Comm_Intl Debt                    Interest Rate                       0   
                                  Grace Period                        0   
                                  Loan Term                           0   
          Equity                  Rate of Return                      0   
                                  Project Life                        0   
          Financing Shares        Debt Share                          0   
                                  Equity Share                        0   
                                  Share of Finance                    0   
          Foreign Currency Shares Debt                                0   
                                  Equity                              0   
Comm_Dom  Debt                    Interest Rate                       0   
                                  Grace Period                        0   
                                  Loan Term                           0   
          Equity                  Rate of Return                      0   
                                  Project Life                        0   
          Financing Shares        Debt Share                          0   
                                  Equity Share                        0   
                                  Share of Finance                    0   
          Foreign Currency Shares Debt                                0   
                                  Equity                              0   
Conc_IFI  Debt                    Interest Rate                       0   
                                  Grace Period                        0   
                                  Loan Term                           0   
          Equity                  Rate of Return                      0   
                                  Project Life                        0   
          Financing Shares        Debt Share                          0   
                                  Equity Share                        0   
                                  Share of Finance                    0   
          Foreign Currency Shares Debt                                0   
                                  Equity                              0   
Conc_DPS  Debt                    Interest Rate                       0   
                                  Grace Period                        0   
                                  Loan Term                           0   
          Equity                  Rate of Return                      0   
                                  Project Life                        0   
          Financing Shares        Debt Share                          0   
                                  Equity Share                        0   
                                  Share of Finance                    0   
          Foreign Currency Shares Debt                                0   
                                  Equity                              0   

                                                     Biomass CSP Geothermal  \
Source    Category                Parameter                                   
Comm_Intl Debt                    Interest Rate     0.269247   0          0   
                                  Grace Period          0.21   0          0   
                                  Loan Term            19.74   0          0   
          Equity                  Rate of Return    0.284547   0          0   
                                  Project Life            30   0          0   
          Financing Shares        Debt Share          0.5344   0          0   
                                  Equity Share        0.4656   0          0   
                                  Share of Finance       0.7   0          0   
          Foreign Currency Shares Debt                     1   0          0   
                               

In [ ]:

# technology_disag_s1_data 是一个行是技术（index），列是MultiIndex（Source, Category, Parameter）的 DataFrame
# 下面举例：取所有技术中，Source=Comm_Intl, Category=Debt, Parameter=Interest Rate 的这一列
# 一个简洁写法：利用 pandas 的 IndexSlice 和 DataFrame 乘法广播，自动对所有 source 批量计算
idx = pd.IndexSlice
fs = technology_disag_s1_data.loc[:, idx[:, 'Financing Shares', :]]
# 先对应相乘再依source/category参数相加，得到每个技术的total份额
# 需要去掉Parameter和Category层，使得两个DataFrame的列索引匹配（都是(Source)），然后对应位置相乘
debt_share = fs.loc[:, idx[:, :, 'Debt Share']].droplevel(['Parameter', 'Category'], axis=1)
share_of_finance = fs.loc[:, idx[:, :, 'Share of Finance']].droplevel(['Parameter', 'Category'], axis=1)
weights_debt = debt_share * share_of_finance

equity_share = fs.loc[:, idx[:, :, 'Equity Share']].droplevel(['Parameter', 'Category'], axis=1)
weights_equity = equity_share * share_of_finance



In [ ]:

technology_disag_s1_data.columns
# technology_disag_s1_data.columns 是一个 pandas MultiIndex
# 查看MultiIndex有哪些属性
attrs = dir(technology_disag_s1_data.columns)

# 每一层是有名字的，用 .names 获取每一层的名字
level_names = technology_disag_s1_data.columns.names

# 取出第一层和第二层的index（实际上是label）
sources = technology_disag_s1_data.columns.get_level_values('Source').unique()
parameters = technology_disag_s1_data.columns.get_level_values('Parameter').unique()
categories = technology_disag_s1_data.columns.get_level_values('Category').unique()
# 比如要取出所有技术的某一个参数，比如"Interest Rate"，


In [ ]:
def cal_weighted_avg(data, weights):
    """统一的加权平均计算函数"""
    return ((weights * data).sum(axis=1) / weights.sum(axis=1).replace(0, np.nan)).fillna(0)

weighted_averages = {}
for category in categories:
    cat = technology_disag_s1_data.loc[:, idx[:, category, :]]
    columns = cat.columns
    debt_mask = columns.get_level_values('Category') == category
    for parameter in columns[debt_mask].get_level_values('Parameter').unique():
        data = cat.loc[:, idx[:, :, parameter]].droplevel(['Parameter', 'Category'], axis=1)
        if category in ["Debt"] or parameter in ["Debt"]:
            weighted_avg = cal_weighted_avg(data, weights_debt)
        elif category in ["Equity"] or parameter in ["Equity"]:
            weighted_avg = cal_weighted_avg(data, weights_equity)
        else:
            if parameter in ["Debt Share"]:
                weighted_avg = weights_debt.sum(axis=1)
            elif parameter in ["Equity Share"]:
                weighted_avg = weights_equity.sum(axis=1)
            else:
                weighted_avg = share_of_finance.sum(axis=1)

        # 使用 (category, parameter) 作为列名
        weighted_averages[(category, parameter)] = weighted_avg


weighted_averages = pd.DataFrame(weighted_averages)
weighted_averages

Debt                          \
                            Interest Rate Grace Period  Loan Term   
Generation Renewable                    0            0          0   
Biomass                          0.182022     1.429992   19.29264   
CSP                                     0            0          0   
Geothermal                              0            0          0   
Hydropower                       0.179978     1.113754  17.558667   
Direct Hydro                            0            0          0   
Nuclear                          0.188404     1.429992   19.29264   
Solar PV                         0.187134     3.011366  23.741785   
Direct Solar                            0            0          0   
Imports                                 0            0          0   
Wind                             0.179794     1.429232  19.289967   
Generation Fossil-Fuel                  0            0          0   
Coal                                    0            0          0   
Gas                              0.247013      0.49973  10.773155   
Direct Oil                              0            0          0   
Oil                                     0            0          0   
Transmission Infrastructure             0            0          0   
Transmission                     0.162176     1.429992   19.29264   
Distribution Infrastructure             0            0          0   
Distribution                     0.162176     1.429992   19.29264   
Exports                                 0            0          0   
Energy Exports                          0            0          0   

                                    Equity              Financing Shares  \
                            Rate of Return Project Life       Debt Share   
Generation Renewable                     0            0                0   
Biomass                           0.280729         30.0         0.663053   
CSP                                      0            0                0   
Geothermal                               0            0                0   
Hydropower                        0.276735    30.379211         0.706322   
Direct Hydro                             0            0                0   
Nuclear                           0.290563         50.0         0.663053   
Solar PV                          0.280711         24.0          0.71189   
Direct Solar                             0            0                0   
Imports                                  0            0                0   
Wind                              0.277292         25.0         0.663062   
Generation Fossil-Fuel                   0            0                0   
Coal                                     0            0                0   
Gas                               0.278581         30.0         0.632759   
Direct Oil                               0            0                0   
Oil                                      0            0                0   
Transmission Infrastructure              0            0                0   
Transmission                      0.250144         50.0         0.663053   
Distribution Infrastructure              0            0                0   
Distribution                      0.250144         70.0         0.663053   
Exports                                  0            0                0   
Energy Exports                           0            0                0   

                                                           \
                            Equity Share Share of Finance   
Generation Renewable                   0                0   
Biomass                         0.336947              1.0   
CSP                                    0                0   
Geothermal                             0                0   
Hydropower                      0.293678              1.0   
Direct Hydro                           0                0   
Nuclear                         0.336947              1.0   
Solar 

In [ ]:
# 假设你的 MultiIndex 存储在 columns 中
columns = technology_disag_s1_data.columns  # 或者你的 MultiIndex

# 获取所有 Category='Debt' 对应的 Parameter 值
# 输出: Index(['Interest Rate', 'Grace Period', 'Loan Term'], dtype='object')

In [ ]:
technologies

[Technology(name='PWRBIO001', tech='Biomass', description='Biomass power plant', classification='Generation Renewable'),
 Technology(name='PWRCOA001', tech='Coal', description='Coal power plant', classification='Generation Fossil-Fuel'),
 Technology(name='PWRCSP001', tech='CSP', description='CSP - Without storage', classification='Generation Renewable'),
 Technology(name='PWRCSP002', tech='CSP', description='CSP - With storage', classification='Generation Renewable'),
 Technology(name='PWRDIST', tech='Distribution', description='Electricity Distribution', classification='Distribution Infrastructure'),
 Technology(name='PWRGEO', tech='Geothermal', description='Geothermal power plant', classification='Generation Renewable'),
 Technology(name='PWRHYD001', tech='Hydropower', description='Hydropower plant - Large dam (>100MW)', classification='Generation Renewable'),
 Technology(name='PWRHYD002', tech='Hydropower', description='Hydropower plant - Medium (10-100MW)', classification='Generati

In [ ]:
import pandas as pd

class InvestmentAllocator:
    """
    负责将投资需求分配到具体的融资渠道。
    """
    def __init__(self, config_df):
        # config_df 是带有 MultiIndex 的 technology_disag_s1_data
        self.config = config_df
        self.idx = pd.IndexSlice

    def allocate(self, 
                 tech_name: str, 
                 total_investment: pd.Series, 
                 source: str,         # 如 'Comm_Intl', 'Conc_IFI'
                 instrument: str,     # 'Debt' 或 'Equity'
                 is_local: bool,      # True 为 Local, False 为 Foreign
                 lc_rate: float, 
                 fc_rate: float) -> pd.Series:
        """
        显式参数分配逻辑，使用向量化计算。
        """
        try:
            # 1. 提取比例参数 (标量)
            sof = self.config.loc[tech_name, self.idx[source, 'Financing Shares', 'Share of Finance']]
            if instrument == 'Debt':
                inst_share = self.config.loc[tech_name, self.idx[source, 'Financing Shares', 'Debt Share']]
                fc_ratio = self.config.loc[tech_name, self.idx[source, 'Foreign Currency Shares', 'Debt']]
            else:
                inst_share = self.config.loc[tech_name, self.idx[source, 'Financing Shares', 'Equity Share']]
                fc_ratio = self.config.loc[tech_name, self.idx[source, 'Foreign Currency Shares', 'Equity']]
            
            # 处理比例中的 NaN
            sof, inst_share, fc_ratio = [x if pd.notna(x) else 0 for x in [sof, inst_share, fc_ratio]]
            
            # 2. 计算当前币种比例 (标量)
            curr_share = (1 - fc_ratio) if is_local else fc_ratio
            
            # 3. 核心：计算汇率因子 (可能是标量，也可能是 Series)
            if is_local:
                # 如果 lc_rate/fc_rate 是 Series，这里会得到一个汇率 Series
                # 使用 .replace(0, np.nan) 避免除以零报错，最后 fillna(0)
                exchange_factor = lc_rate / fc_rate 
            else:
                exchange_factor = fc_rate
            
            # 4. 向量化乘法
            # 如果 total_investment 和 exchange_factor 都是 Series，
            # Pandas 会自动根据索引(Year)进行对齐并对应相乘。
            return total_investment * (sof * inst_share * curr_share) * exchange_factor
            
        except KeyError:
            return total_investment * 0

In [ ]:
from dataclasses import dataclass, field
from typing import Dict, Any,Union ,List
technologies

@dataclass
class SourceConfig:
    # 传入这一行原始数据
    raw_data: Union[pd.Series, Dict] = field(default_factory=dict)
    
    # 定义属性（用于点操作和代码补全）
    fin_share: float = 0.0
    debt_share: float = 0.0
    interest_rate: float = 0.0
    grace_period: float = 0.0
    loan_term: float = 0.0
    rate_of_return: float = 0.0
    project_life: float = 0.0
    fc_debt_ratio: float = 0.0
    fc_equity_ratio: float = 0.0
    
    # 投资结果占位
    inv_fc_debt: pd.Series = None
    inv_fc_equity: pd.Series = None
    inv_lc_debt: pd.Series = None
    inv_lc_equity: pd.Series = None

    def __post_init__(self):
        """对象初始化后自动执行：根据原始数据解析属性"""
        # 定义一个映射表：属性名 -> Excel 中的 (Category, Metric)
        mapping = {
            'fin_share': ('Financing Shares', 'Share of Finance'),
            'debt_share': ('Financing Shares', 'Debt Share'),
            'interest_rate': ('Debt', 'Interest Rate'),
            'grace_period': ('Debt', 'Grace Period'),
            'loan_term': ('Debt', 'Loan Term'),
            'rate_of_return': ('Equity', 'Rate of Return'),
            'project_life': ('Equity', 'Project Life'),
            'fc_debt_ratio': ('Foreign Currency Shares', 'Debt'),
            'fc_equity_ratio': ('Foreign Currency Shares', 'Equity'),
        }
        
        # 遍历映射表，自动赋值
        for attr, key in mapping.items():
            # 从 raw_data 中提取，如果不存在或为 NaN 则设为 0
            val = self.raw_data.get(key, 0.0)
            setattr(self, attr, val if pd.notna(val) else 0.0)

    def __getitem__(self, key):
        """兼容字典访问：config[('Debt', 'Interest Rate')]"""
        return self.raw_data.get(key, 0.0)
    
    def get(self, key, default=0.0):
        """兼容 dict.get()"""
        return self.raw_data.get(key, default)
          
@dataclass
class TechnologyStats(Technology):    
        
    # 这里是年份对应钱数的dict
    name: str
    investment_needs: pd.Series = field(default_factory=lambda: pd.Series(dtype=float))
    revenue: Dict[str, float] = field(default_factory=dict)
    opex: Dict[str, float] = field(default_factory=dict)
    tax: Dict[str, float] = field(default_factory=dict)
    metrics: Dict[str, Any] = field(default_factory=dict)  # 其他自由扩展的指标
    financing_configs: Dict[str, SourceConfig] = field(default_factory=dict)
    tech_df: pd.DataFrame = field(default_factory=lambda: pd.DataFrame())
    def add_metric(self, key: str, value: Any):
        self.metrics[key] = value
        
    def add_financing_source(self, source_name, config_dict):
        self.financing_configs[source_name] = config_dict
        
    def get_allocation_matrix(self, 
                             allocator: InvestmentAllocator, 
                             lc_rate: float, 
                             fc_rate: float):
        """
        一键生成该技术下，所有 Source x Debt/Equity x Local/Foreign 的完整分配矩阵
        """
        # 获取配置表中的所有融资来源名称
        sources = allocator.config.columns.get_level_values('Source').unique()
        
        results = {}
        # 第一层：Currency
        for loc_label, is_local in [('Local Currency', True), ('Foreign Currency', False)]:
            # 第二层：Instrument
            for inst in ['Debt', 'Equity']:
                # 第三层：Source
                for source in sources:
                    # 元组作为 Key，决定了 MultiIndex 的顺序
                    col_key = (loc_label, inst, source)
                    
                    results[col_key] = allocator.allocate(
                        tech_name=self.name,
                        total_investment=self.investment_needs,
                        source=source,
                        instrument=inst,
                        is_local=is_local,
                        lc_rate=lc_rate,
                        fc_rate=fc_rate
                    )
        
        
        df = pd.DataFrame(results)
        df.columns.names = ['Currency', 'Instrument','Source' ]
        return df   
         
    def apply_allocation(self, allocation_df: pd.DataFrame):
        sources = allocation_df.columns.get_level_values('Source').unique()
        for src in sources:
            config = self.financing_configs.get(src)
            if config:
                # 如果发现是字典，现场转成对象
                if isinstance(config, dict):
                    config = SourceConfig(raw_data=config)
                    self.financing_configs[src] = config
                # 现在可以像这样赋值
                config.inv_fc_debt   = allocation_df.get(('Foreign Currency', 'Debt', src), 0)
                config.inv_fc_equity = allocation_df.get(('Foreign Currency', 'Equity', src), 0)
                config.inv_lc_debt   = allocation_df.get(('Local Currency', 'Debt', src), 0)
                config.inv_lc_equity = allocation_df.get(('Local Currency', 'Equity', src), 0)
        


# 方法2：字典 + 按分类分组
from collections import defaultdict

tech_stats_dict = {}
tech_stats_by_class = defaultdict(list)

for tech in technologies:
    tech_stats = TechnologyStats(tech)
    # print(tech)
    tech_stats.investment = df_category_sum_nz[tech.technology].to_dict()
    sources = technology_disag_s1_data.columns.get_level_values(0).unique()
    for src in sources:
        # 从 MultiIndex 中提取该技术、该渠道的所有变量
        # 对应 Excel 里的那一大堆 XLOOKUP
            # config = SourceConfig（
            #     fin_share= technology_disag_s1_data.loc[tech.technology, (src, 'Financing Shares', 'Share of Finance')],
            #     debt_share= technology_disag_s1_data.loc[tech.technology, (src, 'Financing Shares', 'Debt Share')],
            #     interest_rate= technology_disag_s1_data.loc[tech.technology, (src, 'Debt', 'Interest Rate')],
            #     grace_period= technology_disag_s1_data.loc[tech.technology, (src, 'Debt', 'Grace Period')],
            #     loan_term= technology_disag_s1_data.loc[tech.technology, (src, 'Debt', 'Loan Term')],
            #     equity_return= technology_disag_s1_data.loc[tech.technology, (src, 'Equity', 'Rate of Return')],
            #     project_life= technology_disag_s1_data.loc[tech.technology, (src, 'Equity', 'Project Life')],
            #     fc_debt_ratio= technology_disag_s1_data.loc[tech.technology, (src, 'Foreign Currency Shares', 'Debt')],
            #     fc_equity_ratio= technology_disag_s1_data.loc[tech.technology, (src, 'Foreign Currency Shares', 'Equity')]
            #     _raw_data=
            # ）

        tech_stats.financing_configs[src] = SourceConfig(raw_data=technology_disag_s1_data.loc[tech.technology, src])
        # tech_stats.financing_configs[src] = config
        # tech_stats.add_financing_source(src, config)
        

    tech_stats_dict[tech.technology] = tech_stats
    tech_stats_by_class[tech.classification].append(tech_stats)

biomass_stats = tech_stats_dict['Biomass']

renewable_techs = tech_stats_by_class['Generation Renewable']
# for tech in technologies:
#     tech_stats = TechnologyStats(tech)
#     tech_stats = df_category_sum_nz[tech.technology]
biomass_stats.financing_configs['Comm_Intl'].interest_rate



0.2692468687219313

In [ ]:
# 彻底重置并重新加载配置
for tech in technologies:
    tech_name = tech.technology
    t_stats = tech_stats_dict[tech_name]
    
    # 清空旧的配置
    t_stats.financing_configs = {}
    
    sources = technology_disag_s1_data.columns.get_level_values(0).unique()
    for src in sources:
        try:
            # 明确提取 Series 数据
            row_data = technology_disag_s1_data.loc[tech_name, src]
            
            # 显式包装成对象
            config_obj = SourceConfig(raw_data=row_data)
            
            # 存入字典
            t_stats.financing_configs[src] = config_obj
            
        except KeyError:
            continue

# 验证最后一个
print(f"验证：最后一个存入的类型是 {type(t_stats.financing_configs[src])}")

验证：最后一个存入的类型是 <class '__main__.SourceConfig'>


In the following cell, we form up a function to prepare the calculation of total generation.

In [ ]:
# Get the columns from the electricity production dataframe
from collections import defaultdict

def aggregate_tech_production(df, technologies):
    code_to_name = {t.name: t.technology for t in technologies}
    name_to_cols = defaultdict(list)
    for col in df.columns:
        if col in code_to_name:
            name_to_cols[code_to_name[col]].append(col)
    result = pd.DataFrame({name: df[cols].sum(axis=1) for name, cols in name_to_cols.items()}, index=df.index)
    return result

df_generation_tech_sums = aggregate_tech_production(df_elec_production_nz, technologies)
df_potential_generation_tech_sums = aggregate_tech_production(df_potential_generation_nz, technologies)
df_generation_tech_sums

,Biomass,Coal,CSP,Distribution,Geothermal,Hydropower,Direct Hydro,Gas,Nuclear,Direct Oil,Oil,Solar PV,Direct Solar,Transmission,Energy Exports,Imports,Wind
Year,,,,,,,,,,,,,,,,,
2015,0.001,0,0,35.962576,0,21.2155,0,23.473,0,0,0,0.0152,0,41.66396,0,1.245,0
2016,23,0,0,39.386048,0,20.512,0,26.7665,0,0,0,0.0968,0,45.519155,0,3.215,0
2017,0.0004,0,0,44.673572,0,20.652,0,30.3268,0,0,2.8764,0.152,0,51.504725,0,1.5642,0
2018,0.0004,0,0,49.122633,0,22.1,0,36.4319,0,0,2.6912,0.155,0,56.496975,0,0.7542,0
2019,0.0004,0,0,53.062384,0,26.950305,0,39.1865,0,0,2.1901,0.2531,0,60.880755,0,0.654,0
2020,0.001,0,0,59.306654,0,27.564,0,48.2782,0,0,1.652,0.285,0,67.88111,0,0.352,0
2021,0.001,0,0,64.387126,0,27.5012,0,54.0524,0,0,1.426,0.542,0,73.607425,0,0.2012,0
2022,0.001568,0,0,84.244899,0,30.025,0,54.98,0,0,1.4,22.64752,0,96.193093,0,0.1752,0
2023,0.001568,0,0,89.188341,0,30.955775,0,56.298926,0,0,2,26.589369,0,101.7154,0,0.325,0


```c
(XLOOKUP(C$63,'Financing Baseline'!$D$37:$BL$37,XLOOKUP('High Level Dashboard'!$B$5,'Financing Baseline'!$C$38:$C$47,'Financing Baseline'!$D$38:$BL$47))
/
XLOOKUP(C$63,'Financing Baseline'!$D$37:$BL$37,XLOOKUP('Technology Disag (S1)'!$C108,'Financing Baseline'!$C$38:$C$47,'Financing Baseline'!$D$38:$BL$47)))
```
*Reminder*: the above formula is to find the convert rate at a particular year (C63) between the currency used in *Highlevel Dashboard* and the PPA currency in *Tech Disag*

In [ ]:
import numpy as np
import pandas as pd

# ---------------------------------------------------------------------------
# 1. 段分类常量与辅助函数
# ---------------------------------------------------------------------------

SEGMENT_MAP = {
    "Generation": 1,
    "Transmission": 2,
    "Distribution": 3,
    "Export": 4,
    "Exports": 4,
}

UPSTREAM_CATEGORY_BY_NUM = {
    1: "Generation",
    2: "Transmission",
    3: "Distribution",
    4: "Exports",
}


def _classify_segment(text) -> int:
    """将 Classification 或 Category 转为数值 (Generation=1, Trans=2, Dist=3, Export=4)。"""
    if pd.isna(text) or not str(text).strip():
        return 0
    s = str(text).strip().lower()
    if "generation" in s and "export" not in s:
        return 1
    if "transmission" in s:
        return 2
    if "distribution" in s:
        return 3
    if "export" in s:
        return 4
    return 0


def _get_tech_class(tech_name: str, df_technologies: pd.DataFrame) -> str:
    """从 df_technologies 获取技术的 Classification。"""
    row = df_technologies[df_technologies["Technology"] == tech_name]
    if row.empty:
        return ""
    return row["Classification"].values[0]


def _get_offtaker_share_columns(
    tech_name: str,
    upstream_category: str,
    organized_offtaker: pd.DataFrame,
    tech_dataframes: dict,
) -> list:
    """
    根据 organized_offtaker 和当前 tech 的列名，返回 [(offtaker_name, share_col_name), ...]。
    upstream_category: 上游段名，如 "Generation"。
    """
    mask = (
        organized_offtaker["Category"].str.strip().str.lower() == upstream_category.lower()
    ) & (
        organized_offtaker["Name"].str.strip().str.lower() != "exported from:"
    )
    offtakers = organized_offtaker.loc[mask, ["Name", "Currency"]].drop_duplicates()

    df = tech_dataframes.get(tech_name)
    if df is None:
        return []

    out = []
    for _, row in offtakers.iterrows():
        name, curr = row["Name"], row["Currency"]
        base = (
            str(name)
            .lower()
            .replace(" ", "_")
            .replace(":", "")
            .replace("-", "_")
        )
        share_col = f"{base}_share_{curr}"
        if share_col in df.columns:
            out.append((name, share_col))
    return out

def _get_export_techs(tech_dataframes: dict, df_technologies: pd.DataFrame) -> list:
    """返回 Classification 属于 Exports 的技术名列表（用于 ExportGen）。"""
    return [
        t for t in tech_dataframes
        if _classify_segment(_get_tech_class(t, df_technologies)) == 4
    ]
    
def _compute_export_gen(
    export_techs: list,
    df_generation_tech_sums: pd.DataFrame,
    year: int,
) -> float:
    """ExportGen：所有出口类技术的发电量之和 (PJ → GWh)。"""
    total = 0.0
    for t in export_techs:
        if t in df_generation_tech_sums.columns and year in df_generation_tech_sums.index:
            total += df_generation_tech_sums.loc[year, t]
    return total * 1000 / 3.6


# ---------------------------------------------------------------------------
# 2. 主函数：单年 Generation Purchased（含 offtaker 加权）
# ---------------------------------------------------------------------------


def _slug(name):
    return str(name).lower().replace(" ", "_").replace(":", "").replace("-", "_")


def _empty_result(return_breakdown):
    if return_breakdown:
        return 0.0, 0.0, {}
    return 0.0, 0.0


def _upstream_totals(upstream_techs, tech_dataframes, year):
    """(total_wholesale, total_ppa) 上游批发与 PPA 总量。"""
    tw, tp = 0.0, 0.0
    for t in upstream_techs:
        if year not in tech_dataframes[t].index:
            continue
        td = tech_dataframes[t]
        tw += td.loc[year, "whole_sale_generation"]
        tp += td.loc[year, "ppa_met_generation"] * td.loc[year, "ppa_standard_offtaker_share"]
    return tw, tp


def _weighted_components(tech_df, year, offtaker_share_pairs, total_wholesale, total_ppa):
    """(wholesale_component, ppa_component, breakdown_wholesale, breakdown_ppa)。"""
    if not offtaker_share_pairs:
        return total_wholesale, total_ppa, {"(no shares)": total_wholesale}, {"(no shares)": total_ppa}
    wc, pc, bw, bp = 0.0, 0.0, {}, {}
    for name, share_col in offtaker_share_pairs:
        s = float(tech_df.loc[year, share_col]) if share_col in tech_df.columns else 0.0
        w = s * total_wholesale
        p = s * total_ppa
        wc += w
        pc += p
        bw[name], bp[name] = w, p
    return wc, pc, bw, bp


def _get_fx_to_dash(currency, year, dash_curr, exchange_rates):
    if exchange_rates is None or dash_curr is None or currency not in exchange_rates.columns or dash_curr not in exchange_rates.columns or year not in exchange_rates.index:
        return 1.0
    rx = float(exchange_rates.loc[year, currency])
    rd = float(exchange_rates.loc[year, dash_curr])
    return rd / rx if rx else 0.0


def _get_offtaker_tariff_columns(tech_name, upstream_category, organized_offtaker, tech_dataframes):
    mask = (organized_offtaker["Category"].str.strip().str.lower() == upstream_category.lower()) & (organized_offtaker["Name"].str.strip().str.lower() != "exported from:")
    offtakers = organized_offtaker.loc[mask, ["Name", "Currency"]].drop_duplicates()
    df = tech_dataframes.get(tech_name)
    if df is None:
        return []
    return [
        (r["Name"], f"{_slug(r['Name'])}_share_{r['Currency']}", f"{_slug(r['Name'])}_sale_price_{r['Currency']}", r["Currency"])
        for _, r in offtakers.iterrows()
        if f"{_slug(r['Name'])}_share_{r['Currency']}" in df.columns and f"{_slug(r['Name'])}_sale_price_{r['Currency']}" in df.columns
    ]


def _average_tariff(tech_name, tech_df, year, upstream_techs, upstream_category,
                    total_upstream_wholesale, wholesale_component, ppa_component,
                    tech_dataframes, organized_offtaker, exchange_rates, dash_curr):
    denom = wholesale_component + ppa_component
    if denom <= 0:# or exchange_rates is None or dash_curr is None:
        return 0.0

    num_total = 0.0
    
    # 1) 批发部分：若有 offtaker 的 tariff 列，用 buyer 的 share×tariff×fx
    offtaker_pairs = _get_offtaker_tariff_columns(tech_name, upstream_category, organized_offtaker, tech_dataframes)
    if offtaker_pairs:
        num_wholesale = total_upstream_wholesale * sum(
            (float(tech_df.loc[year, sc]) if sc in tech_df.columns else 0.0)
            * (float(tech_df.loc[year, tc]) if tc in tech_df.columns else 0.0)
            * _get_fx_to_dash(curr, year, dash_curr, exchange_rates)
            for _, sc, tc, curr in offtaker_pairs
        )
    else:
        # 2) 回退：用上游 tech 的 sale_price × whole_sale_generation
        num_wholesale = 0.0
        for t in upstream_techs:
            if year not in tech_dataframes[t].index:
                continue
            td = tech_dataframes[t]
            gwh = td.loc[year, "whole_sale_generation"]
            price = td.loc[year, "sale_price"] if "sale_price" in td.columns else 0.0
            curr = str(td.loc[year, "ppa_currency"]) if "ppa_currency" in td.columns else "USD"
            fx = _get_fx_to_dash(curr, year, dash_curr, exchange_rates)
            num_wholesale += gwh * price * fx

    # 3) PPA 部分：用上游 tech 的 ppa_met × ppa_stand_share × ppa_standard_tariff × fx
    num_ppa = 0.0
    for t in upstream_techs:
        if year not in tech_dataframes[t].index:
            continue
        td = tech_dataframes[t]
        g = td.loc[year, "ppa_met_generation"]
        sh = td.loc[year, "ppa_standard_offtaker_share"]
        price = td.loc[year, "ppa_standard_tariff"]
        curr = str(td.loc[year, "ppa_currency"]) if "ppa_currency" in td.columns else "USD"
        fx = _get_fx_to_dash(curr, year, dash_curr, exchange_rates)
        num_ppa += g * sh * price * fx

    return (num_wholesale + num_ppa) / denom


def compute_generation_purchased(
    tech_name: str,
    year: int,
    tech_dataframes: dict,
    df_generation_tech_sums: pd.DataFrame,
    df_technologies: pd.DataFrame,
    organized_offtaker: pd.DataFrame,
    export_source: str = "Generation",
    return_breakdown: bool = False,
    exchange_rates: pd.DataFrame = None,
    dash_curr: str = "USD",
):
    """GenePurchase (GWh/year) + AverageTariff。return: (gp, tariff) 或 (gp, tariff, breakdown)。"""
    tech_class_num = _classify_segment(_get_tech_class(tech_name, df_technologies))
    upstream_num = tech_class_num - 1
    if upstream_num < 1:
        return _empty_result(return_breakdown)

    upstream_techs = [t for t in tech_dataframes if _classify_segment(_get_tech_class(t, df_technologies)) == upstream_num]
    tech_df = tech_dataframes.get(tech_name)
    if not upstream_techs or tech_df is None:
        return _empty_result(return_breakdown)

    export_techs = _get_export_techs(tech_dataframes, df_technologies)
    if tech_class_num == 4:
        val = _compute_export_gen(export_techs, df_generation_tech_sums, year)
        if return_breakdown:
            return val, 0.0, {"ExportGen": val, "average_tariff": 0.0}
        return val, 0.0

    upstream_category = UPSTREAM_CATEGORY_BY_NUM.get(upstream_num, "")
    offtaker_share_pairs = _get_offtaker_share_columns(tech_name, upstream_category, organized_offtaker, tech_dataframes)
    total_wholesale, total_ppa = _upstream_totals(upstream_techs, tech_dataframes, year)
    wholesale_component, ppa_component, bw, bp = _weighted_components(tech_df, year, offtaker_share_pairs, total_wholesale, total_ppa)

    export_deduction = _compute_export_gen(export_techs, df_generation_tech_sums, year) if tech_class_num == _classify_segment(export_source) + 1 else 0.0
    gene_purchase = wholesale_component + ppa_component - export_deduction
    average_tariff = _average_tariff(tech_name,tech_df, year, upstream_techs, upstream_category, total_wholesale, wholesale_component, ppa_component, tech_dataframes, organized_offtaker, exchange_rates, dash_curr)

    if return_breakdown:
        return gene_purchase, average_tariff, {"wholesale": wholesale_component, "ppa": ppa_component, "export_deduction": export_deduction, "by_offtaker_wholesale": bw, "by_offtaker_ppa": bp, "average_tariff": average_tariff}
    return gene_purchase, average_tariff

def compute_capacity_purchased_and_avg_fee(
    tech_name: str,
    year: int,
    tech_dataframes: dict,
    df_technologies: pd.DataFrame,
    exchange_rates: pd.DataFrame = None,
    dash_curr: str = None,
    cap_gwh_col: str = "ppa_contracted_capacity",
    cap_fee_col: str = "ppa_capacity_fee",
    ppa_currency_col: str = "ppa_currency",
) -> tuple:
    """
    复刻 Excel：同时返回
    1) 容量采购 CapGWh 总和 = SUM(OfftakerCheck * CapGWhArray)
    2) 平均容量费（按容量加权、折算到仪表盘币种）= SUM(OfftakerCheck*CapFee*CapGWh*PPAFX) / SUM(OfftakerCheck*CapGWh)
    Exports 时容量采购=0，平均容量费=0。
    """
    tech_class_num = _classify_segment(_get_tech_class(tech_name, df_technologies))
    if tech_class_num == 4:
        return 0.0, 0.0
    upstream_num = tech_class_num - 1
    if upstream_num < 1:
        return 0.0, 0.0

    def _fx(currency):
        if exchange_rates is None or dash_curr is None or currency not in exchange_rates.columns or dash_curr not in exchange_rates.columns or year not in exchange_rates.index:
            return 1.0
        rx = float(exchange_rates.loc[year, currency])
        rd = float(exchange_rates.loc[year, dash_curr])
        return rd / rx if rx else 0.0

    sum_cap_gwh = 0.0
    sum_fee_times_cap = 0.0

    for t in tech_dataframes:
        if _classify_segment(_get_tech_class(t, df_technologies)) != upstream_num:
            continue
        df = tech_dataframes[t]
        if year not in df.index:
            continue
        if cap_gwh_col not in df.columns or cap_fee_col not in df.columns:
            continue
        gwh = float(df.loc[year, cap_gwh_col])
        fee = float(df.loc[year, cap_fee_col])
        curr = str(df.loc[year, ppa_currency_col]) if ppa_currency_col in df.columns else "USD"
        fx = _fx(curr)
        sum_cap_gwh += gwh
        sum_fee_times_cap += fee * gwh * fx

    if sum_cap_gwh <= 0:
        return sum_cap_gwh, 0.0
    avg_fee = sum_fee_times_cap / sum_cap_gwh
    return sum_cap_gwh, avg_fee

# ---------------------------------------------------------------------------
# 3. 按年份序列计算
# ---------------------------------------------------------------------------


def compute_generation_purchased_series(
    tech_name: str,
    tech_dataframes: dict,
    df_generation_tech_sums: pd.DataFrame,
    df_technologies: pd.DataFrame,
    organized_offtaker: pd.DataFrame,
    years: list = None,
    export_source: str = "Generation",
    return_breakdown: bool = False,
    exchange_rates: pd.DataFrame = None,
    dash_curr: str = None,
):
    """按年份计算 Generation Purchased 与 AverageTariff 序列。"""
    if years is None:
        years = tech_dataframes[tech_name].index.astype(int).tolist() if tech_name in tech_dataframes else []

    gp_list, tariff_list = [], []
    breakdowns = [] if return_breakdown else None

    for y in years:
        res = compute_generation_purchased(
            tech_name, y,
            tech_dataframes, df_generation_tech_sums, df_technologies, organized_offtaker,
            export_source=export_source,
            return_breakdown=return_breakdown,
            exchange_rates=exchange_rates,
            dash_curr=dash_curr,
        )
        if return_breakdown:
            gp_list.append(res[0])
            tariff_list.append(res[1])
            breakdowns.append(res[2])
        else:
            gp_list.append(res[0])
            tariff_list.append(res[1])

    gp_series = pd.Series(gp_list, index=years)
    tariff_series = pd.Series(tariff_list, index=years)

    if return_breakdown:
        return gp_series, tariff_series, breakdowns
    return gp_series, tariff_series

def compute_capacity_purchased_series(
    tech_name: str,
    tech_dataframes: dict,
    df_technologies: pd.DataFrame,
    years: list = None,
    exchange_rates: pd.DataFrame = None,
    dash_curr: str = None,
    cap_gwh_col: str = "ppa_contracted_capacity",
    cap_fee_col: str = "ppa_capacity_fee",
    ppa_currency_col: str = "ppa_currency",
) -> tuple:
    """返回 (capacity_purchased_series, average_capacity_fee_series)。"""
    if years is None:
        years = tech_dataframes[tech_name].index.astype(int).tolist() if tech_name in tech_dataframes else []
    cap_list, fee_list = [], []
    for y in years:
        c, f = compute_capacity_purchased_and_avg_fee(
            tech_name, y, tech_dataframes, df_technologies,
            exchange_rates, dash_curr, cap_gwh_col, cap_fee_col, ppa_currency_col,
        )
        cap_list.append(c)
        fee_list.append(f)
    return pd.Series(cap_list, index=years), pd.Series(fee_list, index=years)

In [ ]:

# 1. 实例化分配器
alloc = InvestmentAllocator(technology_disag_s1_data)



In [ ]:
# def get_tech_context(tech_class, segments_df, data_dict):
#     """
#     根据 tech_class 找到其 Category 以及对应的 Offtaker 数据
#     :param tech_class: 当前处理的技术名称 (如 'National Generation FiT')
#     :param segments_df: 带有 Category 列的 cosumer_segments DataFrame
#     :param data_dict: 存放所有数据的字典 (如 tech_dataframes)
#     :return: (category, offtaker_df)
#     """
#     # 1. 查找匹配的行 (支持模糊匹配，防止 tech_class 也有微小差异)
#     row = organized_offtaker[organized_offtaker['Category'].str.lower().apply(lambda x: x in tech_class.lower())]
#     if row.empty:
#         return None, None
    
#     # 获取分类和 Offtaker 名称
#     category = row['Category'].values[0]
#     offtaker_names = row['Name'].dropna().astype(str).tolist()
   
#     # 将 off taker name 列表转换为 tokens，然后循环查找对应数据
#     offtaker_dfs = []
#     for offtaker_name in offtaker_names:
#         offtaker_name_token = str(offtaker_name).lower().replace(" ", "_")
#         # print(offtaker_name, offtaker_name_token)
#         share_key = offtaker_name_token + '_share'
#         if share_key in data_dict:
#             offtaker_dfs.append(data_dict[share_key])
#         else:
#             continue
#             # print(f"Warning: {share_key} not found in data_dict")
#     if offtaker_dfs:
#         offtaker_df = offtaker_dfs[0] if len(offtaker_dfs) == 1 else pd.concat(offtaker_dfs, axis=1)
#     else:
#         offtaker_df = None
#     return category, offtaker_df
def calc_sale_price(df, er):
    price_cols = [c for c in df.columns if '_sale_price_' in c]
    # print(price_cols)
    # 2. 核心计算：每一项 (价格 * 份额 * 汇率) 构成一列，然后按行 (axis=1) 求和
    df['sale_price'] = pd.DataFrame({
        p: df[p] * df[p.replace('_sale_price_', '_share_')] / er[p.split('_')[-1]]
        for p in price_cols if p.replace('_sale_price_', '_share_') in df.columns
    }).sum(axis=1)
    return df['sale_price']


def flatten_multiindex_columns(df):
    """
    This funtion converts a df wiht multi col indices to a single layer;
    and connect the names of different layer with snake_case
    """
    df = df.copy()
    df.columns = [
        '_'.join([str(level).lower() for level in col if str(level)]).replace(" ","_") 
        if isinstance(col, tuple) else str(col) 
        for col in df.columns.values
    ]
    return df

# tech_offtakers={}
for tech_name, tech_data in all_tech_data.items():
    
    # 调用函数获取上下文
    tech_class = df_technologies.loc[df_technologies['Technology'] == tech_name, 'Classification'].values[0]
    # category, tech_offtakers[tech_name] = get_tech_context(tech_class, organized_offtaker, tech_dataframes[tech_name])
    
    tech_dataframes[tech_name]["total_grant_amount"] = df_category_sum_nz[tech_name]*0.05
    tech_dataframes[tech_name]["ppa_direct_offtaker_share"] = 1 - tech_dataframes[tech_name]["ppa_standard_offtaker_share"]
    tech_dataframes[tech_name]["total_generation"] = df_generation_tech_sums[tech_name]*1000/3.6
    tech_dataframes[tech_name]["ppa_contracted_generation"] = tech_dataframes[tech_name]["total_generation"]
    tech_dataframes[tech_name]["redispatch_compensation"] =  (df_potential_generation_tech_sums[tech_name]-tech_dataframes[tech_name]["total_generation"])*tech_dataframes[tech_name]["redispatch_compensation_price"]
    tech_dataframes[tech_name]['ppa_met_generation'] = np.minimum(
                                                            tech_dataframes[tech_name]["total_generation"], 
                                                            tech_dataframes[tech_name]["ppa_contracted_generation"]
                                                        )
    tech_dataframes[tech_name]['whole_sale_generation'] =  tech_dataframes[tech_name]["total_generation"] - tech_dataframes[tech_name]['ppa_met_generation'] 
    
    tech_dataframes[tech_name]["investment_need"] = df_category_sum_nz[tech_name]-df_category_sum_nz[tech_name]*0.05
    #Debts:
    # debt_comm_intl_local= tech_dataframes[tech_name]["investment_need"]*technology_disag_s1_data
    tech = TechnologyStats(
        name=tech_name, 
        investment_needs=tech_dataframes[tech_name]["investment_need"]
    )
    # 直接一行代码存入原 DataFrame
    tech_dataframes[tech_name]['sale_price'] = calc_sale_price(tech_dataframes[tech_name], exchange_rates)
    tech_dataframes[tech_name]['total_wholesale_revnue'] = tech_dataframes[tech_name]['sale_price']*tech_dataframes[tech_name]['whole_sale_generation'] 
    tech_df = tech_dataframes[tech_name]

    # 计算两类 PPA 收入：标准购电方和直接购电方
    ppa_standard_revenue = tech_df["ppa_met_generation"] * tech_df["ppa_standard_offtaker_share"] * tech_df["ppa_standard_tariff"]
    ppa_direct_revenue = tech_df["ppa_met_generation"] * tech_df["ppa_direct_offtaker_share"] * tech_df["ppa_direct_offtaker_tariff"]

    # 若总发电量大于合约发电量，多出来部分计算罚款，否则为0
    excess_generation = tech_df['total_generation'] - tech_df['ppa_contracted_generation']
    penalty = np.minimum(excess_generation, 0) * tech_df['ppa_penalty_tariff']
    
    total_capacity_fee = tech_df['ppa_contracted_capacity']*tech_df['ppa_capacity_fee']
    rate = 1.0
    # 总 PPA 收入 = 标准方收入 + 直购方收入 + 罚款部分
    tech_df['total_ppa_revenue'] = ppa_standard_revenue + ppa_direct_revenue + penalty+total_capacity_fee*rate
    
    tech_name_row = df_technologies[df_technologies["Technology"] == tech_name]["Name"].values[0]
    tech_df['opex'] = df_opex[tech_name_row]
    
    
    df_result = tech.get_allocation_matrix(alloc, lc_rate=rate, fc_rate=rate)
    tech_stats_dict[tech_name].apply_allocation(df_result)


    df_result_flat = df_result.copy()
    df_result_flat = flatten_multiindex_columns(df_result_flat)

    tech_dataframes[tech_name] = pd.concat([tech_dataframes[tech_name], df_result_flat], axis=1)
    
    tech_stats_dict[tech_name].tech_df = tech_dataframes[tech_name].copy()
    print(tech_stats_dict[tech_name].financing_configs['Comm_Intl'].inv_fc_debt.sum())

biomass_df = tech_dataframes['Biomass']
biomass_df.T.head(60)
# tech_offtakers['Biomass']
# organized_offtaker
biomass_df.T.head(50)


1455.1317530972349
0.0
0.0
12327.85334492481
0.0
12653.69083171425
158185.1474591403
0.0
0.0
85337.41420232446
0.0
2162.813959334436
0.0
0.0
63079.70906788991
45178.043132421924
0.0


,2025,2026,2027,2028,2029,2030,2031,2032,2033,2034,...,2061,2062,2063,2064,2065,2066,2067,2068,2069,2070
total_grant_amount,2.8,2.3,1.8,1.3,0.8,0.3,0.8,1.3,1.8,2.3,...,11.949582,12.546928,13.144274,13.741621,0.0,0.0,0.0,0.0,0.0,0.0
ppa_currency,USD,USD,USD,USD,USD,USD,USD,USD,USD,USD,...,USD,USD,USD,USD,USD,USD,USD,USD,USD,USD
ppa_contracted_generation,34166.666667,0.4355,1111.111111,0.4355,1388.888889,0.4355,15833.333333,0.4355,0.083333,0.083333,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
ppa_standard_offtaker_share,1,1,1,1,1,1,1,1,1,1,...,1,1,1,1,1,1,1,1,1,1
ppa_direct_offtaker_tariff,0.56,0.56,0.56,0.56,0.56,0.56,0.56,0.56,0.56,0.56,...,0,0,0,0,0,0,0,0,0,0
ppa_standard_tariff,0.184653,0.184653,0.184653,0.184653,0.184653,0.184653,0.184653,0.184653,0.184653,0.184653,...,0.184653,0.184653,0.184653,0.184653,0.184653,0.184653,0.184653,0.184653,0.184653,0.184653
ppa_contracted_capacity,150,150,150,150,150,150,150,150,150,150,...,0,0,0,0,0,0,0,0,0,0
ppa_capacity_fee,2,2,2,2,2,2,2,2,2,2,...,0,0,0,0,0,0,0,0,0,0
ppa_penalty_tariff,5,5,5,5,5,5,5,5,5,5,...,0,0,0,0,0,0,0,0,0,0
redispatch_compensation_price,5,5,5,5,5,5,5,5,5,5,...,0,0,0,0,0,0,0,0,0,0


In [ ]:
# 只对 Transmission、Distribution、Energy Exports 算
dash_curr = None
for tech_name in tech_dataframes:
    tech_class_num = _classify_segment(_get_tech_class(tech_name, df_technologies))
    if tech_class_num <= 1:
        # Generation 不需要
        tech_dataframes[tech_name]["power purchased"] = 0
        tech_dataframes[tech_name]["tariff"] = 0
        tech_dataframes[tech_name]["capacity purchased"] = 0
        tech_dataframes[tech_name]["capacity tariff"] = 0
        continue
    gp_series, tariff_series = compute_generation_purchased_series(
        tech_name,
        tech_dataframes, df_generation_tech_sums, df_technologies, organized_offtaker,
        exchange_rates=exchange_rates,
        dash_curr=dash_curr,
    )
    cap_series, avg_fee_series = compute_capacity_purchased_series(
        tech_name,
        tech_dataframes, df_technologies,
        exchange_rates=exchange_rates,
        dash_curr=dash_curr,
    )
    tech_dataframes[tech_name] = tech_dataframes[tech_name].assign(
        power_purchased=gp_series,
        tariff=tariff_series,
        capacity_purchased=cap_series,
        capacity_tariff=avg_fee_series,
        power_purchase_cost=lambda x: x["power_purchased"] * x["tariff"] + x["capacity_purchased"] * x["capacity_tariff"],
        )

In [ ]:
# gp, tariff = compute_generation_purchased_series(
#     "Transmission", tech_dataframes, df_generation_tech_sums,
#     df_technologies, organized_offtaker, dash_curr=exchange_rates['USD'] 
# )
# tariff
# tech_dataframes["Transmission"]["power_purchase_cost"]

In [ ]:
gp_series, tariff_series = compute_generation_purchased_series(
    "Transmission", tech_dataframes, df_generation_tech_sums, df_technologies,
    organized_offtaker
)
cap_series, avg_fee_series = compute_capacity_purchased_series(
    "Transmission", tech_dataframes, df_technologies,
)

In [ ]:
import pandas as pd
import numpy as np


import numpy as np
import pandas as pd

import numpy as np
import pandas as pd
def _calc_debt_logic(inv_series, rate, term, grace, fin_share, years):
    """
    1:1 像素级复刻 Excel 债务公式。
    inv_series: 原始投入序列 (sourcefinFC)
    fin_share: 融资占比
    """
    results = pd.Series(0.0, index=years)
    inv_map = inv_series.to_dict()
    repay_period = term - grace

    for curryear in years:
        # --- PART 1: Principal Portion (本金部分) ---
        # Excel: SUMIFS(sourcefinFC, yearsarray, ">="&(curryear-ROUNDDOWN(term,0)+1), yearsarray, "<="&(curryear-ROUNDUP(grace,0)))
        f_term = int(np.floor(term))
        c_grace = int(np.ceil(grace))
        mask_body = (inv_series.index >= (curryear - f_term + 1)) & (inv_series.index <= (curryear - c_grace))
        sum_ifs_val = inv_series[mask_body].sum()
        
        # 修正项1 (End decimal): XLOOKUP(curryear-ROUNDUP(term)+1, ...) * (term-floor_term)
        c_term = int(np.ceil(term))
        p_end_val = inv_map.get(curryear - c_term + 1, 0.0) * (term - np.floor(term))
        
        # 修正项2 (Grace decimal): XLOOKUP(curryear-floor_grace, ...) * (1-IF(frac==0,1,frac))
        f_grace = int(np.floor(grace))
        g_frac = grace - f_grace
        g_inner_if = g_frac if g_frac != 0 else 1.0
        p_grace_val = inv_map.get(curryear - f_grace, 0.0) * (1.0 - g_inner_if)
        
        # 本金总计 = finshare * (主体 + 修正1 + 修正2) / repay_period
        # (注：Excel 里用的是 -PMT(0, term-grace, SUM)，等同于 SUM/repay_period)
        if repay_period != 0:
            total_p = (fin_share * (sum_ifs_val + p_end_val + p_grace_val)) / repay_period
        else:
            total_p = 0.0

        # --- PART 2: Interest Portion (利息部分) ---
        # 1. 窗口总投入: SUMPRODUCT(sourcefinFC, yearsarray <= curryear, yearsarray >= curryear - term)
        mask_window = (inv_series.index <= curryear) & (inv_series.index >= (curryear - term))
        window_inv_sum = inv_series[mask_window].sum()
        
        # 2. 窗口内已还本金: 模拟那个 grace+1, +2, +3 的步进逻辑
        # step = (curryear - inv_year) - grace
        total_repaid_in_window = 0.0
        for inv_y, val in inv_series[mask_window].items():
            step = (curryear - inv_y) - grace
            # 严格遵守 Excel 条件: step > 0 AND step <= repay_period
            if step > 0:
                repaid_units = min(step, repay_period)
                total_repaid_in_window += (val / repay_period) * repaid_units
        
        # 利息 = finshare * (总投入 - 已还本金) * rate
        total_i = (fin_share * (window_inv_sum - total_repaid_in_window)) * rate

        # 汇总：本金 + 利息
        results[curryear] = total_p + total_i

    return results

def _calc_equity_logic(inv_series, eirr, life, years):
    """Mathematical engine for Equity (Dividends)"""
    flow = pd.Series(0.0, index=years)
    for curr_year in years:
        start_window = curr_year - np.floor(life)
        past_inv = inv_series[(years > start_window) & (years <= curr_year)]
        flow[curr_year] = past_inv.sum() * eirr
    return flow

def calculate_detailed_repayments(tech_stats, er, years = list(range(2025,2071)),target_is_local=False, local_curr_code='GHS'):
    results = {}
    years = pd.Index(years)
    er = er.loc[min(years):]
    lc_rate, fc_rate = er[local_curr_code], er['USD']
    conv_fc_to_target = (lc_rate / fc_rate) if target_is_local else 1.0
    conv_lc_to_target = 1.0 if target_is_local else (fc_rate / lc_rate)
    # print("USD 2 GHS", conv_fc_to_target,"GHS 2 USD", conv_lc_to_target)
    for src, config in tech_stats.financing_configs.items():
        # 获取基础金融参数（利率、期限等）
        def get_cfg(metric, default=0):
            # 兼容你的字典结构
            val = config.get(metric)
            if val is None: # 尝试元组键
                for k, v in config.items():
                    if isinstance(k, tuple) and metric in k: return v
            return val

        # --- 直接使用你刚才赋值的投资 Series ---
        # 贷款还款
        inv_fc_debt = config.inv_fc_debt
        inv_lc_debt = config.inv_lc_debt
        # print(inv_fc_debt)
        if (inv_fc_debt.sum() + inv_lc_debt.sum()) >=0:
            rate, term, grace,fin_share = config.interest_rate, config.loan_term, config.grace_period,config.fin_share
            # print( rate, term, grace )
            repay_fc = _calc_debt_logic(inv_fc_debt, rate, term, grace,fin_share, years) * conv_fc_to_target
            repay_lc = _calc_debt_logic(inv_lc_debt, rate, term, grace,fin_share, years) 
            results[f"Loans ({src})"] = (repay_fc + repay_lc).fillna(0)

        # 股权分红
        inv_fc_equity = config.inv_fc_equity
        inv_lc_equity = config.inv_lc_equity
        # print(src,inv_fc_equity,config.interest_rate)

        if (inv_fc_equity.sum() + inv_lc_equity.sum()) >= 0:
            eirr, life = config.rate_of_return, config.project_life
            
            return_fc = _calc_equity_logic(inv_fc_equity, eirr, life, years) * conv_fc_to_target
            return_lc = _calc_equity_logic(inv_lc_equity, eirr, life, years) 
            results[f"Equity ({src})"] = (return_fc + return_lc).fillna(0)

    return pd.DataFrame(results).T

# er 是你的汇率表，索引是年份
# target_is_local: True 则结果为本币(KES/GHS), False 则结果为外币(USD)

# 为 Biomass 生成明细表
biomass_detail_df = calculate_detailed_repayments(
    tech_stats_dict['Biomass'], 
    er=exchange_rates, 
    target_is_local=False,
    local_curr_code ='GHS'
)
# 为所有技术添加financing requirement明细表
financing_requirement_by_tech = {}
for tech_name, tech_stats_obj in tech_stats_dict.items():
    financing_requirement_by_tech[tech_name] = calculate_detailed_repayments(
        tech_stats_obj,
        er=exchange_rates,
        target_is_local=False,
        local_curr_code='GHS'
    )
    financing_requirement_by_tech[tech_name].loc["Financing Requirement"] = financing_requirement_by_tech[tech_name].sum(axis=0)+tech_dataframes[tech_name]['liabilities'].fillna(0)



# 查看结果 (你会看到 Loans (Comm_Intl) 等每一行的数字)
financing_requirement_by_tech['Biomass'].T.head(30)

,Loans (Comm_Intl),Equity (Comm_Intl),Loans (Comm_Dom),Equity (Comm_Dom),Loans (Conc_IFI),Equity (Conc_IFI),Loans (Conc_DPS),Equity (Conc_DPS),Financing Requirement
2025,4.314315,4.933742,0.041520,0.077356,0.020172,0.021123,0.028411,0.0,12.436638
2026,7.856286,8.986459,0.075708,0.140898,0.036741,0.038474,0.051748,0.0,22.186315
2027,10.436136,12.158150,0.100636,0.190627,0.049709,0.052054,0.118244,0.0,30.105555
2028,12.088158,14.448817,0.116630,0.226543,0.059074,0.061861,0.177467,0.0,36.178548
2029,12.846648,15.858457,0.124018,0.248644,0.074687,0.067896,0.219390,0.0,40.439740
2030,12.745902,16.387072,0.123127,0.256932,0.126712,0.070159,0.244455,0.0,42.954359
2031,13.361041,17.796713,0.129112,0.279034,0.171991,0.076194,0.263248,0.0,47.077333
2032,14.725672,20.087379,0.142329,0.314949,0.210726,0.086002,0.276210,0.0,52.843267
2033,16.805499,23.259070,0.162453,0.364678,0.243117,0.099581,0.301007,0.0,41.235406
2034,19.566227,27.311787,0.189155,0.428221,0.269368,0.116932,0.340372,0.0,48.222062


In [ ]:
organized_offtaker

,Category,Name,Currency
0,Generation,National Generation FiT,USD
1,Generation,National Generation PPA,GHS
2,Generation,MiniGrid Generation,GHS
3,Generation,Mining Generation,GHS
4,Generation,Industry Generation,GHS
5,Transmission,Export Sale Price,USD
6,Distribution,Commercial,USD
7,Distribution,Residential,USD
8,Distribution,Industrial,USD
9,Distribution,Wholesale,USD


In [ ]:

# 2. 实例化技术对象
tech = TechnologyStats(
    name='Biomass', 
    investment_needs=tech_dataframes['Biomass']["investment_need"]
)

# 3. 计算完整矩阵
rate = 1.0
df_result = tech.get_allocation_matrix(alloc, lc_rate=rate, fc_rate=rate)



df_result_flat.head()

,local_currency_debt_comm_intl,local_currency_debt_comm_dom,local_currency_debt_conc_ifi,local_currency_debt_conc_dps,local_currency_equity_comm_intl,local_currency_equity_comm_dom,local_currency_equity_conc_ifi,local_currency_equity_conc_dps,foreign_currency_debt_comm_intl,foreign_currency_debt_comm_dom,foreign_currency_debt_conc_ifi,foreign_currency_debt_conc_dps,foreign_currency_equity_comm_intl,foreign_currency_equity_comm_dom,foreign_currency_equity_conc_ifi,foreign_currency_equity_conc_dps
2025,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2026,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2027,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2028,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2029,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


# High Level Dashboard

From this cell we do the statistics for high level dashboard.
- First we import necesssary modules from MinFin 

In [ ]:
from MinFin.utils import *
from MinFin import high_level_dashboard,EconomicParameters,Scenarios,CapitalInjection

- Then we set up the necessary parameters, by assigning values to several dataclass defined in the MinFin Module

In [ ]:
econ_params = EconomicParameters(
    income_elasticity_of_energy_demand=0.7, 
    cagr_of_real_energy_price=0.01, 
    gdp_growth_rate=0.0526
)

capital_injection =  CapitalInjection(
    money_from = "domestic public",
    start_year = 2024,
    volume = 100.0,
    duration = 3,
    type = "loan"
    )

scenarios = Scenarios(
    financial_instrument_type = "Grant",
    scenario = "NetZero",
    capital_injection = capital_injection,
)
        

In [ ]:
@dataclass
class DisagLookupConfig:
    financing_source: str = ""
    variable_type: str = ""

- Updates: we add technology specific waccs here:


In [ ]:
from dataclasses import dataclass
from typing import Optional
import pandas as pd


@dataclass
class DisagLookupConfig:
    scenario: str = "S1"
    financing_source: str = ""


def build_disag_table(
    config: DisagLookupConfig,
    disag_data: pd.DataFrame,
    tech_list: Optional[list] = None,
) -> pd.DataFrame:
    """提取该 Source 下全部列，保留 (Category, Parameter) MultiIndex，缺值 NA。"""
    src = config.financing_source
    tech_list = tech_list or list(disag_data.index)

    if not src:
        return pd.DataFrame(index=tech_list)

    try:
        block = disag_data.loc[:, (src, slice(None), slice(None))].copy()
        block.columns = block.columns.droplevel("Source")
    except (KeyError, TypeError):
        block = pd.DataFrame(index=disag_data.index)

    return block.reindex(tech_list)

cfg = DisagLookupConfig(financing_source="Comm_Intl")

df = build_disag_table(cfg, technology_disag_s1_data)
df

Category                             Debt                         \
Parameter                   Interest Rate Grace Period Loan Term   
Generation Renewable                    0            0         0   
Biomass                          0.269247         0.21     19.74   
CSP                                     0            0         0   
Geothermal                              0            0         0   
Hydropower                       0.265747         0.21     19.74   
Direct Hydro                            0            0         0   
Nuclear                          0.279247         0.21     19.74   
Solar PV                         0.265747         0.21     19.74   
Direct Solar                            0            0         0   
Imports                                 0            0         0   
Wind                             0.265747         0.21     19.74   
Generation Fossil-Fuel                  0            0         0   
Coal                                    0            0         0   
Gas                              0.263547     0.204981  9.996187   
Direct Oil                              0            0         0   
Oil                                     0            0         0   
Transmission Infrastructure             0            0         0   
Transmission                     0.238147         0.21     19.74   
Distribution Infrastructure             0            0         0   
Distribution                     0.238147         0.21     19.74   
Exports                                 0            0         0   
Energy Exports                          0            0         0   

Category                            Equity              Financing Shares  \
Parameter                   Rate of Return Project Life       Debt Share   
Generation Renewable                     0            0                0   
Biomass                           0.284547           30           0.5344   
CSP                                      0            0                0   
Geothermal                               0            0                0   
Hydropower                        0.281047           30           0.5962   
Direct Hydro                             0            0                0   
Nuclear                           0.294547           50           0.5344   
Solar PV                          0.281047           24           0.5962   
Direct Solar                             0            0                0   
Imports                                  0            0                0   
Wind                              0.281047           25           0.5344   
Generation Fossil-Fuel                   0            0                0   
Coal                                     0            0                0   
Gas                               0.278847           30           0.5962   
Direct Oil                               0            0                0   
Oil                                      0            0                0   
Transmission Infrastructure              0            0                0   
Transmission                      0.253447           50           0.5344   
Distribution Infrastructure              0            0                0   
Distribution                      0.253447           70           0.5344   
Exports                                  0            0                0   
Energy Exports                           0            0                0   

Category                                                   \
Parameter                   Equity Share Share of Finance   
Generation Renewable                   0                0   
Biomass                           0.4656              0.7   
CSP                                    0                0   
Geothermal                             0                0   
Hydropower                        0.4038              0.7   
Direct Hydro                           0                0   
Nuclear                           0.4656              

In [ ]:
hd = high_level_dashboard(repayment_statistics,econ_params,scenarios,df_funding_baseline_full=df_funding_baseline_full,least_cost_summary=least_cost_summary,net_zero_summary=net_zero_summary)

### Example usage of the class

# hd.get_funding_availability_lever()
# hd.get_co2_savings(least_cost_summary,net_zero_summary)
hd.get_financing_source_shares()

,Historical,Projected
Conc_IFI,0.419847,0.419847
Conc_DPS,0.111346,0.111346
Comm_Intl,0.414019,0.414019
Comm_Dom,0.054789,0.054789


In [ ]:
nz_needs = hd.get_net_zero_financing_needs_full(df_invest_need_summary,df_funding_envelope) # Net Zero finacning needs
lc_needs = hd.get_least_cost_financing_needs_full(df_invest_need_summary,df_funding_envelope) # Least Cost finacning needs
additional_investment_needs = hd.get_additional_investment_needs(lc_needs,nz_needs) #Additional needs (NetZero - LeastCost)
# nz_needs.head(6)


KeyError: "None of [Index(['LeastCost', 'Total'], dtype='object')] are in the [columns]"

In [ ]:
df_invest_need_summary 
# df_funding_envelope

,Net Zero,Least Cost,Incremental,Total financing
Year,,,,
2025,6498.979201,882.56605,5616.413151,6498.979201
2026,10500.692932,373.29072,10127.402212,10500.692932
2027,9535.663318,640.41796,8895.245358,9535.663318
2028,12333.219742,256.21112,12077.008622,12333.219742
2029,22531.171126,251.91853,22279.252596,22531.171126
2030,42086.07542,247.62594,41838.44948,42086.07542
2031,40750.626594,578.022367,40172.604227,40750.626594
2032,30966.418357,754.489211,30211.929146,30966.418357
2033,37908.254391,1144.10409,36764.150301,37908.254391
